# 00. 제품 키워드 통합 파이프라인

**목적**: 4개 소스의 키워드를 정제하여 HIN 학습용 소스 데이터 생성

**최종 출력물**

| 파일 | 용도 |
|------|------|
| `insta_keywords_processed.parquet` | Instagram 제품 키워드 소스 |
| `blog_keywords_processed.parquet` | 블로그 제품 키워드 소스 |
| `trend_keywords_processed.parquet` | 트렌드 커뮤니티 키워드 소스 |
| `ip_master_dataset.parquet` | IP 마스터 (ip_name\|일반_속성\|트렌드_속성\|소속_커뮤니티) |
| `instagram_engagement_with_keywords.parquet` | 편의점명\|원본명\|정규화명\|키워드\|좋아요 수\|언급일\|url\|body |
| `trend_engagement_with_keywords.parquet` | 트렌드(키워드)\|속성보유여부\|속성(키워드)\|좋아요 수\|언급일\|url\|body |
| `pos_blog_keywords_bridge.parquet` | 원본명(POS명)\|정규화명\|키워드 (수동 검수용) |

**실행 순서**: Phase 0→0B→0C → Phase 1A/B/B-2/C → Phase 3 → Phase 4/4C → Phase 2D → Phase 4D/4E → Phase 5-3

> *(수동 검수용 셀은 주석처리 상태로 보존. 재검수 시 해당 셀 활성화.)*

> ※ `final_product_keywords.parquet` 및 `product_master_dataset.parquet` 생성은 `01_pos_feature_engineering.ipynb` 에서 수행


## Phase 0. 환경 설정

라이브러리 임포트, 경로 설정, `keyword_rules.py` 로드, 4-step 정제 함수 정의


In [1]:
import pandas as pd
import numpy as np
import os
import re
import sys
import json
import ast
import pickle
import importlib
import difflib
import warnings
from collections import OrderedDict, Counter

warnings.filterwarnings('ignore')

BASE_DIR = r'C:\Users\송정현\Documents\Projects\박재홍교수님세미나\Projects\20기\7eleven_npd_framework'

RAW_INSTA_PATH  = os.path.join(BASE_DIR, 'data', 'processed', '편의점_instagram', 'merged_instagram_products_final.xlsx')
LLM_RESULT_PATH = os.path.join(BASE_DIR, 'data', 'processed', '편의점_instagram', 'smart_clean_result.xlsx')
COMPARE_XLSX    = os.path.join(BASE_DIR, 'eda', 'df_compare_keywords.xlsx')
B4_RAW_PATH     = os.path.join(BASE_DIR, 'data', 'processed', 'B4_ITEM_DV_INFO_with_survival.csv')
B4_PATH         = os.path.join(BASE_DIR, 'data', 'processed', 'B4_ITEM_DV_INFO_with_survival.csv')
OUTPUT_PATH     = os.path.join(BASE_DIR, 'data', 'processed', 'final_product_keywords.parquet')
CHECKPOINT_PATH = os.path.join(BASE_DIR, 'eda', 'df_qual_checkpoint.pkl')

INSTA_PROC_PATH  = os.path.join(BASE_DIR, 'data', 'processed', 'insta_keywords_processed.parquet')
BLOG_PROC_PATH   = os.path.join(BASE_DIR, 'data', 'processed', 'blog_keywords_processed.parquet')
TREND_PROC_PATH  = os.path.join(BASE_DIR, 'data', 'processed', 'trend_keywords_processed.parquet')

B4_FILTERED_PATH = os.path.join(BASE_DIR, 'data', 'processed', 'B4_ITEM_DV_INFO_filtered.parquet')
POS_PATH         = os.path.join(BASE_DIR, 'data', 'processed', 'POS 전처리 최종', 'pos_data_food_final_상품단위변환전.parquet')
PARETO_XLSX      = os.path.join(BASE_DIR, 'eda', '카테고리필터링_pareto기준자르기.xlsx')

print('경로 설정 완료')
print(f'BASE_DIR = {BASE_DIR}')


경로 설정 완료
BASE_DIR = C:\Users\송정현\Documents\Projects\박재홍교수님세미나\Projects\20기\7eleven_npd_framework


In [2]:
# ─── keyword_rules.py 로드 ───────────────────────────────────────
sys.path.insert(0, os.path.join(BASE_DIR, 'eda'))
import keyword_rules
importlib.reload(keyword_rules)           # 파일 변경 시 재실행 없이 반영

REMOVE_KWS         = set(keyword_rules.REMOVE_KWS)
SYNONYM_MAP        = dict(keyword_rules.SYNONYM_MAP)
SPLIT_MAP          = dict(keyword_rules.SPLIT_MAP)
SANDWICH_PRODUCTS  = set(keyword_rules.SANDWICH_PRODUCTS)
CONTAINS_COLLAPSE  = list(keyword_rules.CONTAINS_COLLAPSE)
EXTRA_SPLIT_MAP    = dict(keyword_rules.EXTRA_SPLIT_MAP)
EXTRA_SYNONYM_MAP  = dict(keyword_rules.EXTRA_SYNONYM_MAP)
REMOVE_WORDS       = set(keyword_rules.REMOVE_WORDS)
KEYWORD_MAPPING    = dict(keyword_rules.KEYWORD_MAPPING)

# ─── PATCH 병합 ──────────────────────────────────────────────────
REMOVE_KWS     |= keyword_rules.PATCH_REMOVE
SYNONYM_MAP.update(keyword_rules.PATCH_SYNONYM)
SPLIT_MAP.update(keyword_rules.PATCH_SPLIT)
KEYWORD_MAPPING.update(keyword_rules.PATCH_DECOMPOSE)

# ─── 프로모션 코드 병합 ──────────────────────────────────────────
SYNONYM_MAP.update({
    kw: code
    for code, kws in keyword_rules.PROMO_MAP.items()
    for kw in kws
})

print('keyword_rules.py 로드 완료')
print(f'  REMOVE_KWS:      {len(REMOVE_KWS)}개')
print(f'  SYNONYM_MAP:     {len(SYNONYM_MAP)}개 (프로모션 포함)')
print(f'  SPLIT_MAP:       {len(SPLIT_MAP)}개')
print(f'  KEYWORD_MAPPING: {len(KEYWORD_MAPPING)}개')

keyword_rules.py 로드 완료
  REMOVE_KWS:      164개
  SYNONYM_MAP:     643개 (프로모션 포함)
  SPLIT_MAP:       143개
  KEYWORD_MAPPING: 162개


In [3]:
# ── 4-step 키워드 정제 파이프라인 함수 정의 ──────────────────────
def remove_noise_kws(kw_list):
    if not isinstance(kw_list, list): return kw_list
    return [kw for kw in kw_list if kw not in REMOVE_KWS]

def unify_keywords(kw_list):
    if not isinstance(kw_list, list): return kw_list
    result = []
    for kw in kw_list:
        if kw in SPLIT_MAP:
            result.extend(SPLIT_MAP[kw])
        else:
            result.append(SYNONYM_MAP.get(kw, kw))
    return list(dict.fromkeys(result))

def apply_advanced_rules(kw_list, product_name=''):
    if not isinstance(kw_list, list): return kw_list
    result = []
    for kw in kw_list:
        if kw == '제로':
            result.extend(['제로', '카페인'] if product_name == '제로모히또제로카페인' else ['제로', '슈거'])
            continue
        if kw == '샌드':
            result.append('샌드위치' if product_name in SANDWICH_PRODUCTS else '샌드')
            continue
        if kw in EXTRA_SPLIT_MAP:
            result.extend(EXTRA_SPLIT_MAP[kw]); continue
        collapsed = next((w for w in CONTAINS_COLLAPSE if w in kw), None)
        if collapsed:
            result.append(collapsed); continue
        if kw in REMOVE_WORDS: continue
        result.append(EXTRA_SYNONYM_MAP.get(kw, kw))
    return list(dict.fromkeys(result))

def apply_keyword_mapping(kw_list):
    if isinstance(kw_list, str):
        try:    kw_list = ast.literal_eval(kw_list)
        except: kw_list = [kw_list]
    if not isinstance(kw_list, list): return []
    expanded = []
    for kw in kw_list:
        expanded.extend(KEYWORD_MAPPING.get(kw, [kw]))
    return list(OrderedDict.fromkeys(expanded))

def run_pipeline(kw_list, product_name=''):
    kw_list = remove_noise_kws(kw_list)
    kw_list = unify_keywords(kw_list)
    kw_list = apply_advanced_rules(kw_list, product_name)
    kw_list = apply_keyword_mapping(kw_list)
    return kw_list

print('파이프라인 함수 정의 완료')

파이프라인 함수 정의 완료


### Phase 0B. B4 카테고리 필터링

`B4_ITEM_DV_INFO_with_survival.csv` 원본에서 분석 대상 상품군만 추출 → `B4_ITEM_DV_INFO_filtered.parquet` 저장
- **대분류 필터**: 식품 관련 22개 카테고리
- **중분류 필터**: 파레토 기준 통과 카테고리 (`카테고리필터링_pareto기준자르기.xlsx` 사용여부='O')
- 이후 모든 셀은 이 필터링된 파일을 B4 마스터로 사용

In [4]:
# ── Phase 0B: B4 카테고리 필터링 → B4_ITEM_DV_INFO_filtered.parquet ─
FOOD_CATEGORIES = [
    '음료', '과자', '유음료', '미반', '면', '냉장', '맥주', '즉석음료', '빵', '전통주',
    '아이스크림', '조리빵', '즉석 식품', '건강/기호식품', '가공식품', '양주와인', '디저트',
    '안주', '신선', '간식', '조미료/건물', '냉동',
]

if os.path.exists(B4_FILTERED_PATH):
    print(f'B4 필터링 파일 이미 존재 → 로드: {B4_FILTERED_PATH}')
    _b4_check = pd.read_parquet(B4_FILTERED_PATH)
    print(f'  상품 수: {len(_b4_check):,}개 / 중분류: {_b4_check["ITEM_MDDV_NM"].nunique()}개')
else:
    _b4_raw = pd.read_csv(B4_PATH, dtype={'ITEM_CD': str})
    print(f'B4 원본 로드: {len(_b4_raw):,}개')

    # 1. 대분류 필터
    _b4_food = _b4_raw[_b4_raw['ITEM_LRDV_NM'].isin(FOOD_CATEGORIES)].copy()
    print(f'대분류 필터 후: {len(_b4_food):,}개')

    # 2. 파레토 중분류 필터
    _pareto = pd.read_excel(PARETO_XLSX)
    _survived = set(_pareto.loc[_pareto['사용여부'] == 'O', 'ITEM_MDDV_NM'])
    _b4_filtered = _b4_food[_b4_food['ITEM_MDDV_NM'].isin(_survived)].copy()
    print(f'파레토 필터 후: {len(_b4_filtered):,}개 / 중분류: {_b4_filtered["ITEM_MDDV_NM"].nunique()}개')

    _b4_filtered.to_parquet(B4_FILTERED_PATH, index=False)
    print(f'저장 완료 → {B4_FILTERED_PATH}')

# 이후 모든 셀에서 B4_PATH = 필터링된 파일
B4_PATH = B4_FILTERED_PATH
print(f'\nB4_PATH → 필터링된 파일로 갱신 완료')

B4 필터링 파일 이미 존재 → 로드: C:\Users\송정현\Documents\Projects\박재홍교수님세미나\Projects\20기\7eleven_npd_framework\data\processed\B4_ITEM_DV_INFO_filtered.parquet
  상품 수: 49,957개 / 중분류: 51개

B4_PATH → 필터링된 파일로 갱신 완료


### Phase 0C. NPD 플래그 계산

**목적**: B2 POS 데이터 기준 burn-in 14일 이후 첫 등장 상품에 is_npd=True 플래그 부여  
**입력**: POS_PATH (영업일자, 상품코드) + B4_ITEM_DV_INFO_filtered.parquet  
**출력**: B4_ITEM_DV_INFO_filtered.parquet (is_npd 컬럼 추가, 행 수 동일)  

- burn_in_cutoff = 데이터셋 최초 영업일 + 14일
- 첫 판매일 > cutoff → is_npd = True
- POS에 미등장 상품 → is_npd = False


In [5]:
# ── Phase 0C: is_npd 플래그 계산 → B4_ITEM_DV_INFO_filtered 업데이트 ─
BURN_IN_DAYS = 14

print("[Phase 0C] POS 데이터 로드 중...")
_pos = pd.read_parquet(POS_PATH, columns=["영업일자", "상품코드"])
_pos["영업일자"] = pd.to_datetime(_pos["영업일자"].astype(str), format="%Y%m%d")

dataset_start  = _pos["영업일자"].min()
burn_in_cutoff = dataset_start + pd.Timedelta(days=BURN_IN_DAYS)
print(f"  데이터셋 시작일: {dataset_start.date()} / burn-in 기준일: {burn_in_cutoff.date()}")

_first_sale = (_pos.groupby("상품코드")["영업일자"]
               .min()
               .rename("첫판매일")
               .reset_index()
               .rename(columns={"상품코드": "ITEM_CD"}))

_b4 = pd.read_parquet(B4_FILTERED_PATH)
_b4 = _b4.merge(_first_sale, on="ITEM_CD", how="left")
_b4["is_npd"] = (_b4["첫판매일"] >= burn_in_cutoff).fillna(False)  # >= : 2025-01-15 당일 포함
_b4 = _b4.drop(columns=["첫판매일"])

npd_count   = int(_b4["is_npd"].sum())
total_count = len(_b4)
print(f"  전체: {total_count:,}개 | NPD: {npd_count:,}개 ({npd_count/total_count*100:.1f}%) | 비NPD: {total_count-npd_count:,}개")

_b4.to_parquet(B4_FILTERED_PATH, index=False)
print(f"  parquet 저장 → {B4_FILTERED_PATH}")

_b4_csv_path = B4_FILTERED_PATH.replace(".parquet", ".csv")
_b4.to_csv(_b4_csv_path, index=False, encoding="utf-8-sig")
print(f"  CSV 저장 → {_b4_csv_path}")

del _pos, _first_sale, _b4


[Phase 0C] POS 데이터 로드 중...
  데이터셋 시작일: 2025-01-01 / burn-in 기준일: 2025-01-15
  전체: 49,957개 | NPD: 3,128개 (6.3%) | 비NPD: 46,829개
  parquet 저장 → C:\Users\송정현\Documents\Projects\박재홍교수님세미나\Projects\20기\7eleven_npd_framework\data\processed\B4_ITEM_DV_INFO_filtered.parquet
  CSV 저장 → C:\Users\송정현\Documents\Projects\박재홍교수님세미나\Projects\20기\7eleven_npd_framework\data\processed\B4_ITEM_DV_INFO_filtered.csv


## Phase 1. 소스별 키워드 전처리

인스타그램 / 블로그 / 트렌드 각 소스가 독립적으로 4-step 파이프라인을 거쳐 별도 parquet로 저장됨

| 소스 | 입력 | 출력 |
|------|------|------|
| Instagram | `merged_instagram_products_final.xlsx` | `insta_keywords_processed.parquet` |
| 블로그 | `블로그_키워드_품질분석_데이터셋_분류완료.csv` | `blog_keywords_processed.parquet` |
| 트렌드 | `찐최종뉴뉴_community_1hop_mapping_final.csv` | `trend_keywords_processed.parquet` |

### [선택] 세븐일레븐 smart_clean 결과 병합

 실행 후 검수 완료 시:
 →  로 rename
그 후 아래 셀을 실행하면 에 병합됩니다.

In [6]:
# smart_clean_result_seven_final.xlsx 존재 시에만 병합 실행
SEVEN_FINAL_PATH = os.path.join(
    BASE_DIR, 'data', 'processed', '편의점_instagram', 'smart_clean_result_seven_final.xlsx'
)

if not os.path.exists(SEVEN_FINAL_PATH):
    print("⏭️  smart_clean_result_seven_final.xlsx 없음 → 병합 건너뜀")
else:
    df_seven = pd.read_excel(SEVEN_FINAL_PATH)
    df_comp  = pd.read_excel(COMPARE_XLSX)

    existing_names = set(df_comp['제품명'].astype(str))
    df_new = df_seven[~df_seven['제품명'].astype(str).isin(existing_names)].copy()

    if len(df_new) == 0:
        print("⏭️  추가할 신규 제품 없음 (이미 모두 병합됨)")
    else:
        df_new = df_new.rename(columns={
            '정제_전_키워드': '이전 키워드',
            '확정_키워드':   '확정 키워드',
        })
        df_new = df_new[['제품명', '이전 키워드', '확정 키워드', '생존후보_키워드']]

        df_merged = pd.concat([df_comp, df_new], ignore_index=True)
        df_merged.to_excel(COMPARE_XLSX, index=False)

        print(f"✅ 병합 완료: {len(df_new)}개 세븐 제품 추가 → 총 {len(df_merged)}개")
        print(f"   추가된 제품 ({len(df_new)}개):")
        for name in df_new['제품명'].tolist():
            print(f"     · {name}")


⏭️  추가할 신규 제품 없음 (이미 모두 병합됨)


### Phase 1A. Instagram 키워드 전처리

Step 2~5.8: 속성 파싱 → 필터링 → 제품명 정규화 → 중복 병합 → LLM 통합 → 복구 → IP 콜라보 매핑 → 최종키워드 확정
- **출력**: `insta_keywords_processed.parquet`

In [7]:
# ── Step 2. 기초 속성 파싱 ──────────────────────────────────────
df = pd.read_excel(RAW_INSTA_PATH)
print(f'원천 데이터 로드: {df.shape}')

def safe_parse_attrs(val):
    if pd.isna(val) or val == '[]' or val == '': return []
    try:
        if isinstance(val, str):
            try: return json.loads(val)
            except: return ast.literal_eval(val)
        return val
    except: return []

df['p_attrs_list'] = df['p_attrs'].apply(safe_parse_attrs)

STOPWORDS_CATEGORIZED = {
    'Contextual': {'이벤트','참여','댓글','팔로우','당첨','경품','사전예약','선착순','증정','할인','행사','원플러스원','투플러스원','1+1','2+1','공식','계정','어플','앱','포켓CU','우리동네GS','없음'},
    'Marketing':  {'신상','신제품','NEW','추천','인기','대박','출시','한정판','한정','단독','주목','달려가세요','쟁여두세요','필수','박스','기획','패키지','에디션','컬렉션','시리즈'},
    'Channel':    {'세븐일레븐','CU','GS25','씨유','지에스','편의점','세븐','편의점신상'},
    'Subjective': {'맛있다','예쁘다','좋아요','강추','비주얼','꿀맛','존맛','미쳤다','역대급','JMT'},
}
ALL_STOPWORDS = set().union(*STOPWORDS_CATEGORIZED.values())

def clean_keywords(attrs):
    if not isinstance(attrs, list): return []
    cleaned = []
    for kw in attrs:
        kw = str(kw).replace(' ', '').strip()
        if kw in ALL_STOPWORDS or len(kw) <= 1: continue
        if re.search(r'\d+(ml|g|kg|l|개|입|봉|팩|병|캔)', kw, re.I): continue
        cleaned.append(kw)
    return list(set(cleaned))

df['p_attrs_cleaned'] = df['p_attrs_list'].apply(clean_keywords)

# ── Step 3. 원본 기준 필터링 ─────────────────────────────────────
contains_exclude = ['오늘의 메뉴', 'PBICK', '거강기능식품', '장건강', '&']
exact_exclude    = ['탄산음료','삼각김밥','도시락','김밥','주먹밥','샌드위치','햄버거','와인','스프린트','청년다방','김치']
mask = (df['p_name'].str.contains('|'.join(contains_exclude), na=False, case=False)
      | df['p_name'].isin(exact_exclude))
df = df[~mask].reset_index(drop=True)
print(f'필터링 후: {len(df)}개 제품')

# ── Step 4. 제품명 정규화 + 중복 병합 ───────────────────────────
def normalize_product_name(name):
    if not isinstance(name, str): return name
    name = re.sub(r'\[.*?\]|\(.*?\)', '', name)
    for word in ['신상','한정판','2\\+1','1\\+1','증정','단독','출시','NEW']:
        name = name.replace(word, '')
    name = re.sub(r'[^a-zA-Z0-9가-힣\s]', ' ', name)
    return re.sub(r'\s+', ' ', name).strip()

df['p_name_clean'] = df['p_name'].apply(normalize_product_name)

def merge_unique_keywords(series):
    merged = []
    for k in series:
        if isinstance(k, list): merged.extend(k)
    return list(set(merged))

agg = {col: 'first' for col in df.columns if col not in ['p_name', 'p_attrs_cleaned']}
agg['p_attrs_cleaned'] = merge_unique_keywords
df = df.groupby('p_name', as_index=False).agg(agg)
print(f'중복 병합 후: {len(df)}개 고유 제품')

# ── Step 5. Surgical Clean ──────────────────────────────────────
REMOVE_PARTS   = ['각종', '의맛']
STRICT_TARGETS = ['경주', '고소', '짭잘', '골든', '공부', '안유성']

def surgical_clean(attrs):
    if not isinstance(attrs, list): return []
    res = []
    for kw in attrs:
        kw = str(kw).strip()
        hit = next((t for t in STRICT_TARGETS if t in kw), None)
        if hit:
            res.append(hit)
            continue
        for part in REMOVE_PARTS: kw = kw.replace(part, '')
        if len(kw.strip()) > 1: res.append(kw.strip())
    return list(set(res))

df['p_attrs_cleaned'] = df['p_attrs_cleaned'].apply(surgical_clean)
print('기초 전처리 완료')

원천 데이터 로드: (5099, 10)
필터링 후: 4947개 제품
중복 병합 후: 3765개 고유 제품
기초 전처리 완료


In [8]:
# ── Step 5.5. LLM 결과 통합 ─────────────────────────────────────
if os.path.exists(LLM_RESULT_PATH):
    df_res = pd.read_excel(LLM_RESULT_PATH)
    def to_list(val):
        if pd.isna(val) or str(val).strip() == '': return []
        return [w.strip() for w in str(val).split(',')]
    df_res['p_attrs_confirmed'] = df_res['확정_키워드'].apply(to_list)
    df_res['p_attrs_rescued']   = df_res['생존후보_키워드'].apply(to_list)
    df_m = df.merge(df_res[['제품명','p_attrs_confirmed','p_attrs_rescued']],
                    left_on='p_name', right_on='제품명', how='left')
    df['p_attrs_before']    = df['p_attrs_cleaned']
    df['p_attrs_confirmed'] = df_m['p_attrs_confirmed'].apply(lambda x: x if isinstance(x, list) else [])
    df['p_attrs_rescued']   = df_m['p_attrs_rescued'].apply(lambda x: x if isinstance(x, list) else [])
    df['p_attrs_cleaned']   = df['p_attrs_confirmed']
    # 세븐 브랜드 중 LLM 미검수(confirmed=[])는 p_attrs_before 사용
    _mask = (df['brand'] == '세븐') & (df['p_attrs_confirmed'].apply(len) == 0)
    df.loc[_mask, 'p_attrs_cleaned'] = df.loc[_mask, 'p_attrs_before']
    print(f'LLM 결과 병합 완료: {len(df)}개 제품 (세븐 fallback: {_mask.sum()}개)')
else:
    df['p_attrs_before']    = df['p_attrs_cleaned']
    df['p_attrs_confirmed'] = df['p_attrs_cleaned']
    df['p_attrs_rescued']   = [[] for _ in range(len(df))]
    print('LLM 파일 없음 — 원본 키워드로 진행')

# ── Step 6. 복구 로직 (restore logic) ───────────────────────────
MUST_RESTORE_FROM_BEFORE = [
    '가나디','불고기','물만두','매드포갈릭','마른안주','파인애플','게맛살','샌드위치','콩나물','닭고기',
    '군만두','시리얼','알룰로스','급식대가','베이커리','프랑스','이모카세','떡갈비','피넛버터','라즈베리',
    '오리온','오징어게임','올리브유','보양식','아이스크림','당충전','흑백요리사','칼국수','디저트39','편다이닝',
    '이웃집통통이','고독한미식가더무비','아메리카노','우유니소금','짱구는못말려','콘치즈','포켓몬스터','버터베어',
    '시트러스','애플하우스','역전우동','황치즈','리얼프라이스','에드워드리','밀크티','차돌박이','타코야끼',
    '투다리','소주전쟁','앙버터','아사이볼',
]
MUST_RESCUE_WORDS = ['밥','감귤','쏘이','궁채','퓨전','야식','제철','술','유튜버','봄','빵']

def restore_logic(row):
    base    = list(row['p_attrs_confirmed']) if isinstance(row['p_attrs_confirmed'], list) else []
    before  = list(row['p_attrs_before'])    if isinstance(row['p_attrs_before'],    list) else []
    rescued = list(row['p_attrs_rescued'])   if isinstance(row['p_attrs_rescued'],   list) else []
    for kw in MUST_RESTORE_FROM_BEFORE:
        if kw in before: base.append(kw)
    for kw in MUST_RESCUE_WORDS:
        if kw in before or kw in rescued: base.append(kw)
    return list(set(base))

df['p_attrs_cleaned'] = df.apply(restore_logic, axis=1)
print('restore logic 완료')

LLM 결과 병합 완료: 3765개 제품 (세븐 fallback: 835개)
restore logic 완료


In [9]:
# ── Step 5.7. IP 콜라보 맵 ───────────────────────────────────────
# 프로모션 맵은 keyword_rules.PROMO_MAP → SYNONYM_MAP으로 병합됨 (kw-rules-load 셀)
_collab_all = [
    ('게임_IP',    '블루 아카이브',  ['블루아카','블루아카이브','몰루','아로나']),
    ('게임_IP',    '메이플스토리',   ['메이플스토리','메이플','핑크빈','예티','주황버섯','슬라임']),
    ('게임_IP',    '승리의여신니케', ['니케','NIKKE','시프트업']),
    ('K팝_아이돌', '세븐틴',         ['세븐틴','SVT','SEVENTEEN','캐럿']),
    ('K팝_아이돌', 'PLAVE',          ['플레이브','PLAVE','예준','노아','밤비','은호','하민']),
    ('콘텐츠_IP',  '흑백요리사',     ['흑백요리사','요리계급전쟁','백수저','흑수저','급식대가','나폴리맛피아','이모카세','안성재','백종원']),
    ('게임_IP',    'T1',             ['T1','페이커','FAKER','제오페구케']),
    ('K팝_아이돌', 'TXT',            ['투모로우바이투게더','TXT','투바투','MOA']),
    ('K팝_아이돌', '지드래곤',       ['피스마이너스원','지드래곤','GD','권지용','PMO','데이지']),
    ('캐릭터_IP',  '포켓몬스터',     ['포켓몬','포켓몬스터','피카츄','메타몽','띠부씰']),
    ('캐릭터_IP',  '짱구',           ['짱구','짱구는못말려','흰둥이','짱아','초코비','못말려']),
    ('캐릭터_IP',  '가나디',         ['가나디','GANADI']),
    ('캐릭터_IP',  '빵빵이',         ['빵빵이','옥지']),
    ('F&B_브랜드', '백종원',         ['백종원','백주부','더본코리아']),
    ('F&B_브랜드', '오징어게임',     ['오징어게임','성기훈']),
    ('셰프_IP',    '에드워드리',     ['에드워드리']),
    ('방송_IP',    '정희원',         ['정희원']),
    ('캐릭터_IP',  '산리오',         ['산리오','헬로키티','페코짱']),
    ('캐릭터_IP',  '디즈니',         ['디즈니','주토피아','미키','랏소']),
]
ip_search_map = {}
for cat, ip_name, kws in _collab_all:
    for kw in kws:
        ip_search_map[kw] = {'ip': ip_name, 'cat': cat}
for chef in ['최강록','박은영','안유성','에드워드리','이균']:
    ip_search_map[chef] = {'ip': ['흑백요리사', chef], 'cat': '셰프_IP'}

print(f'ip_search_map: {len(ip_search_map)}개')

ip_search_map: 79개


In [10]:
# ── Step 5.8. 최종 키워드 확정 ─────────────────────────────────
CONTAINS_REPLACE = [
    (r'수건',                                    ['수건', '케이크']),
    (r'망곰',                                    ['망곰']),
    (r'두산|베어스',                              ['KBO']),
    (r'요거트아이스정석|요거트아이스크림의정석',   ['요아정']),
]
EXACT_REPLACE = {'삼각': ['삼각김밥'], '삼각김밥': ['삼각김밥']}
PATTERN_ALLOWLIST = {
    r'티':       {'노티드','링티','밀크티','스파게티','짜파게티','아이스티','캐치!티니핑','티라미수','블랙티'},
    r'아이스':   {'아이스크림','아이스브륄레'},
    r'데이':     {'데이지에일','데이트','발렌타인데이','블랙데이','빼빼로데이','화이트데이'},
    r'.*베리$':  {'라즈베리','블루라즈베리','블루베리','스트로베리','아사이베리','크랜베리'},
    r'.*고기$':  {'불고기','돼지고기','닭고기','머릿고기','소고기','쇠고기','오리고기'},
    r'.*가루$':  {'고춧가루','김가루','들깨가루','콩가루'},
    r'배달':     {'배달','픽업'},
    r'1인':      {'1인'},
    r'빵':       {'깨찰빵','꽃빵','맘모스빵','미각제빵소','붕어빵','빵또아','빵빵이','소금빵','소보로빵','중화빵','호빵'},
}
REMOVE_PATTERNS = [r'셰프']

def parse_kw_col(val):
    if pd.isna(val) or str(val).strip() == '': return []
    return [w.strip() for w in str(val).split(',') if w.strip()]

def apply_rules(kw, is_confirmed=False):
    kw       = str(kw).strip()
    kw_clean = kw.replace(' ', '')
    if kw_clean in ip_search_map:
        v = ip_search_map[kw_clean]['ip']
        return v if isinstance(v, list) else [v]
    for pat in REMOVE_PATTERNS:
        if re.search(pat, kw): return []
    for pat, repl in CONTAINS_REPLACE:
        if re.search(pat, kw): return repl
    if kw in EXACT_REPLACE: return EXACT_REPLACE[kw]
    # 프로모션 치환은 run_pipeline → SYNONYM_MAP에서 처리
    for pat, allowlist in PATTERN_ALLOWLIST.items():
        if re.search(pat, kw):
            return [kw] if kw in allowlist else []
    return [kw] if is_confirmed else []

def build_final_keywords(row):
    def to_kw(val):
        if isinstance(val, list): return val
        return parse_kw_col(val)
    confirmed = to_kw(row.get('확정 키워드',  row.get('p_attrs_confirmed', [])))
    before    = to_kw(row.get('이전 키워드',  row.get('p_attrs_before',    [])))
    rescued   = to_kw(row.get('생존후보_키워드', row.get('p_attrs_rescued', [])))
    result = []
    for kw in confirmed:
        result.extend(apply_rules(kw, is_confirmed=True))
    for kw in list(dict.fromkeys(before + rescued)):
        if kw not in confirmed:
            result.extend(apply_rules(kw, is_confirmed=False))
    return list(dict.fromkeys(result))

if os.path.exists(COMPARE_XLSX):
    df_qual = pd.read_excel(COMPARE_XLSX)
    print(f'수동 검수 대조표 로드: {len(df_qual)}개 제품')
    # COMPARE_XLSX에 없는 제품(세븐 등) 보충 — p_attrs_cleaned(fallback 적용됨)로 채움
    _existing = set(df_qual['제품명'])
    _df_new = df[~df['p_name'].isin(_existing)].copy()
    if len(_df_new) > 0:
        _rows = pd.DataFrame({
            '제품명':          _df_new['p_name'].values,
            '이전 키워드':     _df_new['p_attrs_before'].apply(lambda x: ', '.join(x) if isinstance(x, list) else ''),
            '확정 키워드':     _df_new['p_attrs_cleaned'].apply(lambda x: ', '.join(x) if isinstance(x, list) else ''),
            '생존후보_키워드': _df_new['p_attrs_rescued'].apply(lambda x: ', '.join(x) if isinstance(x, list) else ''),
        })
        df_qual = pd.concat([df_qual, _rows], ignore_index=True)
        print(f'미검수 신규 제품 {len(_rows)}개 보충 (fallback 키워드 적용) → 총 {len(df_qual)}개')
else:
    df_qual = pd.DataFrame({
        '제품명':          df['p_name'].values,
        '이전 키워드':     df['p_attrs_before'].apply(lambda x: ', '.join(x) if isinstance(x, list) else ''),
        '확정 키워드':     df['p_attrs_confirmed'].apply(lambda x: ', '.join(x) if isinstance(x, list) else ''),
        '생존후보_키워드': df['p_attrs_rescued'].apply(lambda x: ', '.join(x) if isinstance(x, list) else ''),
    })
    print(f'df_compare 인메모리 생성: {len(df_qual)}개 제품')

df_qual['확정키워드_최종'] = df_qual.apply(build_final_keywords, axis=1)
print(f'확정키워드_최종 생성 완료 — 평균 {df_qual["확정키워드_최종"].apply(len).mean():.1f}개/제품')

수동 검수 대조표 로드: 3789개 제품
미검수 신규 제품 31개 보충 (fallback 키워드 적용) → 총 3820개
확정키워드_최종 생성 완료 — 평균 6.3개/제품


In [11]:
df_meta = df[['p_name','brand','p_price','p_cap','date']].drop_duplicates('p_name')
if '제품명' not in df_qual.columns and 'p_name' in df_qual.columns:
    df_qual = df_qual.rename(columns={'p_name': '제품명'})
df_qual = df_qual.merge(df_meta, left_on='제품명', right_on='p_name', how='left').drop(columns='p_name', errors='ignore')

print(f'df_qual 완성: {len(df_qual)}개 제품')
print(f'컬럼: {list(df_qual.columns)}')
df_qual[['제품명','brand','p_price','p_cap','확정키워드_최종']].head(5)

df_qual 완성: 3820개 제품
컬럼: ['제품명', '이전 키워드', '확정 키워드', '생존후보_키워드', '확정키워드_최종', 'brand', 'p_price', 'p_cap', 'date']


,제품명,brand,p_price,p_cap,확정키워드_최종
0,100%두리안바,GS25,6900.0,40G*3입,"[간식, 두리안, 스낵, 프리미엄, 한정판매]"
1,1000트위스트,GS25,0.0,85G,"[골든, 디저트, 떡볶이, 로제, 마요, 매콤, 밀크, 반찬, 비빔, 서울, 소다,..."
2,1664블랑캔,GS25,0.0,NaN,"[맥주, 묶음할인, 쟁여두기]"
3,1865청뱀띠에디션,CU,31900.0,NaN,"[1865, 과일, 바닐라, 시즌, 와인, 청뱀띠, 특별할인]"
4,1988 버거,GS25,0.0,NaN,"[1988, 도시락, 시즌, 야식, 코카, 콜라, 콤보]"


In [12]:
# ── Block 2 완료: Instagram 파이프라인 적용 + 저장 ─────────────
df_qual['확정키워드_정제'] = df_qual['확정키워드_최종'].apply(run_pipeline)

df_insta_proc = df_qual[['제품명', '확정키워드_정제', 'p_price', 'p_cap', 'brand']].copy()
df_insta_proc.to_parquet(INSTA_PROC_PATH, index=False, engine='pyarrow')
print(f'Instagram 전처리 저장: {INSTA_PROC_PATH}')
print(f'shape: {df_insta_proc.shape}')

Instagram 전처리 저장: C:\Users\송정현\Documents\Projects\박재홍교수님세미나\Projects\20기\7eleven_npd_framework\data\processed\insta_keywords_processed.parquet
shape: (3820, 5)


In [13]:
# ── Block 2 키워드 현황 (파이프라인 적용 후) ─────────────────────
_kw  = df_qual['확정키워드_정제'].apply(lambda x: len(x) if isinstance(x, list) else 0)
_all = [kw for row in df_qual['확정키워드_정제'] if isinstance(row, list) for kw in row]
print('=== Block 2 완료: Instagram 키워드 파이프라인 ===')
print(f'  상품 수          : {len(df_qual):,}개')
print(f'  총 키워드 수     : {_kw.sum():,}개')
print(f'  고유 키워드 수   : {len(set(_all)):,}개')
print(f'  평균 키워드 수   : {_kw.mean():.1f}개/상품')
print(f'  키워드 없는 상품 : {(_kw == 0).sum():,}개')
print(f'저장: {INSTA_PROC_PATH}')

=== Block 2 완료: Instagram 키워드 파이프라인 ===
  상품 수          : 3,820개
  총 키워드 수     : 23,727개
  고유 키워드 수   : 1,894개
  평균 키워드 수   : 6.2개/상품
  키워드 없는 상품 : 34개
저장: C:\Users\송정현\Documents\Projects\박재홍교수님세미나\Projects\20기\7eleven_npd_framework\data\processed\insta_keywords_processed.parquet


### Phase 1B. 블로그 키워드 전처리

**입력**: `data/raw/블로그_키워드_품질분석_데이터셋_분류완료.csv` (euc-kr)
- `관련있음` 컬럼의 검증된 키워드 파싱 → 4-step 파이프라인 적용
- ITEM_CD 매칭 없음 (제품명 기반 join은 Phase 4에서 수행)
- **출력**: `blog_keywords_processed.parquet`

In [14]:
# ── Block 4B: 블로그 파이프라인 + 저장 ──────────────────────────
BLOG_CSV = os.path.join(BASE_DIR, 'data', 'raw', '블로그_키워드_품질분석_데이터셋_분류완료.csv')

if os.path.exists(BLOG_CSV):
    df_blog_raw = pd.read_csv(BLOG_CSV, encoding='euc-kr')

    def parse_blog_kws(val):
        if pd.isna(val) or str(val).strip() == '':
            return []
        return [w.strip() for w in str(val).split(',') if w.strip()]

    df_blog_raw['blog_kws_raw']    = df_blog_raw['관련있음'].apply(parse_blog_kws)
    df_blog_raw['확정키워드_정제'] = df_blog_raw['blog_kws_raw'].apply(run_pipeline)

    df_blog_proc = df_blog_raw[['상품명', '확정키워드_정제']].copy()
    df_blog_proc.to_parquet(BLOG_PROC_PATH, index=False, engine='pyarrow')

    _all_blog = [kw for lst in df_blog_proc['확정키워드_정제'] if isinstance(lst, list) for kw in lst]
    print(f'블로그 전처리 완료: {len(df_blog_proc)}개 상품')
    print(f'  고유 키워드: {len(set(_all_blog))}개')
    print(f'저장: {BLOG_PROC_PATH}')
else:
    pd.DataFrame(columns=['상품명', '확정키워드_정제']).to_parquet(BLOG_PROC_PATH, index=False, engine='pyarrow')
    print(f'⚠️ 블로그 파일 없음: {BLOG_CSV}')
    print(f'빈 파일 저장: {BLOG_PROC_PATH}')

블로그 전처리 완료: 2011개 상품
  고유 키워드: 1696개
저장: C:\Users\송정현\Documents\Projects\박재홍교수님세미나\Projects\20기\7eleven_npd_framework\data\processed\blog_keywords_processed.parquet


### Phase 1B-2. 블로그 재추출 + 신규 크롤링 병합

**목적**: 재추출 결과와 신규 크롤링 결과를 에 병합  
**입력**:
-  (기존 재추출본, 있으면 병합)
-  ( 출력, 있으면 병합)

**출력**:  (나중 데이터 우선, 중복 상품명 제거)

---

In [15]:
# ── Phase 1B-2: 블로그 재추출 + 신규 크롤링 병합 ──────────────────
RECRAWL_PATH   = os.path.join(BASE_DIR, 'data', 'processed', 'blog_keywords_recrawl_final.csv')
NEW_CRAWL_PATH = os.path.join(BASE_DIR, 'data', 'processed', 'blog_keywords_new_crawl.parquet')

def _parse_numpy_kw(v):
    """numpy 배열 문자열 → list[str] (예: ['식사' '도시락'] → ['식사','도시락'])"""
    if isinstance(v, list):
        return v
    return re.findall(r"'([^']+)'", str(v))

_base = pd.read_parquet(BLOG_PROC_PATH)
before = len(_base)
_parts = [_base]

# ① 기존 재추출본 (blog_keywords_recrawl_final.csv)
if os.path.exists(RECRAWL_PATH):
    _reex = pd.read_csv(RECRAWL_PATH, encoding='utf-8-sig')
    _reex = _reex.rename(columns={'원본명': '상품명'})
    _reex['확정키워드_정제'] = _reex['확정키워드_정제'].apply(_parse_numpy_kw)
    _reex = _reex[['상품명', '확정키워드_정제']]
    _parts.append(_reex)
    print(f'  기존 재추출본(csv): {len(_reex)}개')
else:
    print(f'  기존 재추출본(csv) 없음 → skip')

# ② 신규 크롤링 결과 (batch_blog_new_crawl.py 출력)
if os.path.exists(NEW_CRAWL_PATH):
    _new = pd.read_parquet(NEW_CRAWL_PATH)
    _new = _new.rename(columns={'ITEM_NM': '상품명'})[['상품명', '확정키워드_정제']]
    _parts.append(_new)
    print(f'  신규 크롤링(parquet): {len(_new)}개')
else:
    print(f'  신규 크롤링(parquet) 없음 → skip')

_merged = (
    pd.concat(_parts, ignore_index=True)
    .drop_duplicates(subset='상품명', keep='last')  # 나중 것 우선
    .reset_index(drop=True)
)
_merged.to_parquet(BLOG_PROC_PATH, index=False, engine='pyarrow')
print(f'[Phase 1B-2] 병합 완료: 기존 {before}개 → {len(_merged)}개')
del _base, _parts, _merged

  기존 재추출본(csv): 346개
  신규 크롤링(parquet): 537개
[Phase 1B-2] 병합 완료: 기존 2011개 → 2883개


### Phase 1C. 트렌드 키워드 전처리

**입력**: `찐최종뉴뉴_community_1hop_mapping_final.csv` + `제품_속성.json`
- `키워드_분류결과_속성` 전체 값 + `제품_속성.json` keywords → 4-step 파이프라인 → trend_flag_kws 부여 → IP 연결 군집 집계
- **출력**: `trend_keywords_processed.parquet`

In [16]:
# ── Block 4C: 트렌드 파이프라인 + trend_flag_kws + 저장 ─────────
TREND_PATH        = os.path.join(BASE_DIR, 'data', 'processed', 'IP_속성추출',
                                 '찐최종뉴뉴_community_1hop_mapping_final.csv')
PRODUCT_ATTR_JSON = os.path.join(BASE_DIR, 'data', 'processed', 'IP_속성추출', '제품_속성.json')

df_trend_csv = pd.read_csv(TREND_PATH, encoding='utf-8-sig')
df_trend_csv = df_trend_csv.rename(columns={
    'IP/브랜드/캐릭터/인플루언서 키워드': 'ip_col',
    '키워드_분류결과_제품명':             '트렌드_제품명',
    '키워드_분류결과_속성':              '트렌드_속성원본',
})
print(f'트렌드 CSV 로드: {df_trend_csv.shape}')

def parse_comma_str(val):
    if pd.isna(val) or str(val).strip() in ('', '없음'):
        return []
    return [v.strip() for v in str(val).split(',') if v.strip()]

def normalize_trend_name(name):
    return re.sub(r'\s+', '', str(name).strip()).lower()

# ── 제품_속성.json 로드 ──────────────────────────────────────────
_prod_attrs_raw = []
product_attr_map = {}
if os.path.exists(PRODUCT_ATTR_JSON):
    with open(PRODUCT_ATTR_JSON, 'r', encoding='utf-8') as f:
        _prod_attrs_raw = json.load(f)
    for item in _prod_attrs_raw:
        kw = normalize_trend_name(item.get('keyword', ''))
        attrs = (
            item.get('flavor', []) +
            item.get('texture', []) +
            item.get('ingredients', []) +
            item.get('tpo', [])
        )
        product_attr_map[kw] = [a for a in attrs if isinstance(a, str) and a.strip()]
    print(f'제품_속성.json 로드: {len(product_attr_map)}개 제품명')
else:
    print('⚠️ 제품_속성.json 없음 — 제품명 추론 속성 없이 진행')

# ── trend_kw_set 구축 ────────────────────────────────────────────
# 소스 1: 키워드_분류결과_속성 전체 값 → 파이프라인 처리
_attr_all = [kw for val in df_trend_csv['트렌드_속성원본'] for kw in parse_comma_str(val)]
_attr_processed = run_pipeline(_attr_all)

# 소스 2: 제품_속성.json keyword 필드값 → 파이프라인 처리
_prod_kw_raw = [normalize_trend_name(item.get('keyword', ''))
                for item in _prod_attrs_raw if item.get('keyword')]
_prod_kw_processed = run_pipeline([k for k in _prod_kw_raw if k])

trend_kw_set = set(_attr_processed) | set(_prod_kw_processed)
print(f'trend_kw_set 구축: {len(trend_kw_set)}개 키워드')
# trend_kw_set 저장 → 01에서 동일 기준 사용
_trend_kw_set_path = os.path.join(BASE_DIR, 'data', 'processed', 'trend_kw_set.json')
with open(_trend_kw_set_path, 'w', encoding='utf-8') as _f:
    json.dump({'trend_kw_set': sorted(trend_kw_set)}, _f, ensure_ascii=False)
print(f'trend_kw_set 저장: {_trend_kw_set_path} ({len(trend_kw_set)}개)')

# ── 트렌드 제품명별 행 생성 + 파이프라인 + trend_flag_kws ────────
trend_rows = []
for _, row in df_trend_csv.iterrows():
    prod_names   = parse_comma_str(row['트렌드_제품명'])
    attr_kws_raw = parse_comma_str(row['트렌드_속성원본'])
    ip_list      = parse_comma_str(row.get('ip_col', ''))

    for prod_name in prod_names:
        norm_name = normalize_trend_name(prod_name)
        inferred  = product_attr_map.get(norm_name, [])
        combined  = list(dict.fromkeys(attr_kws_raw + inferred))
        processed = run_pipeline(combined)

        trend_rows.append({
            '트렌드_제품명':   prod_name,
            '확정키워드_정제': processed,
            'ip_list':         ip_list,
            '군집ID':          str(row.get('군집ID', '')),
            '군집명':          row.get('군집명', ''),
        })

df_trend_proc = pd.DataFrame(trend_rows)

# ── 군집별 trend_flag_kws 통합 (Issue 2 해결) ──────────────────
# 군집 내 모든 제품의 키워드 중 trend_kw_set에 속하는 것을 합쳐서 군집 시그니처로 정의
cluster_flags = (
    df_trend_proc.explode('확정키워드_정제')
    .query('확정키워드_정제 in @trend_kw_set')
    .groupby('군집ID')['확정키워드_정제']
    .apply(lambda x: sorted(list(set(x))))
    .to_dict()
)
df_trend_proc['trend_flag_kws'] = df_trend_proc['군집ID'].map(cluster_flags).apply(lambda x: x if isinstance(x, list) else [])

df_trend_proc.to_parquet(TREND_PROC_PATH, index=False, engine='pyarrow')

_all_trend = [kw for lst in df_trend_proc['확정키워드_정제'] if isinstance(lst, list) for kw in lst]
_flag_kws  = [kw for lst in df_trend_proc['trend_flag_kws']  if isinstance(lst, list) for kw in lst]
print(f'트렌드 전처리 완료: {len(df_trend_proc)}개 제품명 (행)')
print(f'  고유 키워드       : {len(set(_all_trend))}개')
print(f'  고유 트렌드 플래그: {len(set(_flag_kws))}개')
print(f'저장: {TREND_PROC_PATH}')

트렌드 CSV 로드: (22, 14)
제품_속성.json 로드: 447개 제품명
trend_kw_set 구축: 747개 키워드
trend_kw_set 저장: C:\Users\송정현\Documents\Projects\박재홍교수님세미나\Projects\20기\7eleven_npd_framework\data\processed\trend_kw_set.json (747개)
트렌드 전처리 완료: 448개 제품명 (행)
  고유 키워드       : 732개
  고유 트렌드 플래그: 381개
저장: C:\Users\송정현\Documents\Projects\박재홍교수님세미나\Projects\20기\7eleven_npd_framework\data\processed\trend_keywords_processed.parquet


## Phase 2. 키워드 어휘 검토 Export *(수동 검수용 — 재검수 시 활성화)*

> 최초 1회 실행 후 수동 검수 완료. 재검수가 필요할 때만 아래 셀을 활성화.


In [17]:
# [수동 검수용 — 필요 시 주석 해제]
# # ── 키워드 어휘 전수 검토 Excel export ──────────────────────────
# # Block 2/4B/4C 완료 후 → 3소스 전체 확정키워드_정제 어휘 목록
# VOCAB_REVIEW_PATH = os.path.join(BASE_DIR, 'data', 'processed', 'keyword_vocab_review.xlsx')
#
# def collect_vocab(parquet_path, source_name, kw_col='확정키워드_정제'):
#     if not os.path.exists(parquet_path):
#         print(f'  ⚠️ 파일 없음: {parquet_path}')
#         return {}
#     df_ = pd.read_parquet(parquet_path)
#     if kw_col not in df_.columns:
#         print(f'  ⚠️ 컬럼 없음: {kw_col}')
#         print(f'     존재하는 컬럼: {list(df_.columns)}')
#         return {}
#     sample = df_[kw_col].iloc[0] if len(df_) > 0 else None
#     print(f'  {source_name}: {len(df_)}행, dtype={df_[kw_col].dtype}, sample={repr(sample)[:80]}')
#
#     def to_list(val):
#         if isinstance(val, list):
#             return val
#         try:
#             return list(val)  # pyarrow list, ndarray 등 iterable 처리
#         except TypeError:
#             pass
#         if isinstance(val, str):
#             try:
#                 return ast.literal_eval(val)
#             except Exception:
#                 return []
#         return []
#
#     counter = Counter(
#         kw
#         for lst in df_[kw_col].apply(to_list)
#         for kw in lst
#         if isinstance(kw, str) and kw
#     )
#     print(f'     → 고유 키워드: {len(counter)}개')
#     return {kw: {'빈도': cnt, source_name: True} for kw, cnt in counter.items()}
#
# vocab_insta = collect_vocab(INSTA_PROC_PATH, '인스타')
# vocab_blog  = collect_vocab(BLOG_PROC_PATH,  '블로그')
# vocab_trend = collect_vocab(TREND_PROC_PATH, '트렌드')
#
# # 전체 어휘 합산
# all_kws = set(vocab_insta) | set(vocab_blog) | set(vocab_trend)
# print(f'\n전체 고유 키워드: {len(all_kws)}개')
#
# rows = []
# for kw in sorted(all_kws):
#     total = (vocab_insta.get(kw, {}).get('빈도', 0)
#            + vocab_blog.get(kw,  {}).get('빈도', 0)
#            + vocab_trend.get(kw, {}).get('빈도', 0))
#     rows.append({
#         '키워드':     kw,
#         '총빈도':     total,
#         '인스타':     '✓' if kw in vocab_insta else '',
#         '블로그':     '✓' if kw in vocab_blog  else '',
#         '트렌드':     '✓' if kw in vocab_trend else '',
#         '인스타빈도': vocab_insta.get(kw, {}).get('빈도', 0),
#         '블로그빈도': vocab_blog.get(kw,  {}).get('빈도', 0),
#         '트렌드빈도': vocab_trend.get(kw, {}).get('빈도', 0),
#     })


In [18]:
# [수동 검수용 — 필요 시 주석 해제]
# df_vocab = pd.DataFrame(rows).sort_values('총빈도', ascending=False).reset_index(drop=True)
# df_vocab.to_excel(VOCAB_REVIEW_PATH, index=False)
#
# print(f'어휘 검토 파일 저장: {VOCAB_REVIEW_PATH}')
# print(f'  전체 고유 키워드: {len(df_vocab):,}개')
# print(f'  인스타 전용    : {((df_vocab["인스타"]=="✓") & (df_vocab["블로그"]=="") & (df_vocab["트렌드"]=="")).sum():,}개')
# print(f'  블로그 전용    : {((df_vocab["블로그"]=="✓") & (df_vocab["인스타"]=="") & (df_vocab["트렌드"]=="")).sum():,}개')
# print(f'  트렌드 전용    : {((df_vocab["트렌드"]=="✓") & (df_vocab["인스타"]=="") & (df_vocab["블로그"]=="")).sum():,}개')
# print(f'  3소스 공통     : {((df_vocab["인스타"]=="✓") & (df_vocab["블로그"]=="✓") & (df_vocab["트렌드"]=="✓")).sum():,}개')
# print(f'\n상위 20개:')
# print(df_vocab[['키워드','총빈도','인스타','블로그','트렌드']].head(20).to_string(index=False))


## Phase 3. IP 속성 키워드 처리

**입력**: `IP_속성.json` (전체) + `IP_속성_재추출.json` (재추출분, 있으면 override)
- Step 2: 두 파일 병합 → `IP_속성_통합.json`
- Step 3: `signature_keywords` 고유값 export → `ip_keyword_vocab_review.xlsx` (수동 검수용)
- Step 4: `ip_keyword_vocab_review_final.xlsx` 반영 → `IP_키워드사전.json` 생성

In [19]:
# ── Step 2. IP_속성.json + IP_속성_재추출.json 병합 ──────────────
IP_ATTR_JSON     = os.path.join(BASE_DIR, 'data', 'processed', 'IP_속성추출', 'IP_속성.json')
IP_RERUN_JSON    = os.path.join(BASE_DIR, 'data', 'processed', 'IP_속성추출', 'IP_속성_재추출.json')
IP_MERGED_JSON   = os.path.join(BASE_DIR, 'data', 'processed', 'IP_속성추출', 'IP_속성_통합.json')
IP_KW_VOCAB_PATH = os.path.join(BASE_DIR, 'data', 'processed', 'ip_keyword_vocab_review.xlsx')

with open(IP_ATTR_JSON, 'r', encoding='utf-8') as f:
    base_records = json.load(f)
base_map = {r['keyword']: r for r in base_records}
print(f'기존 IP_속성.json: {len(base_map)}개')

if os.path.exists(IP_RERUN_JSON):
    with open(IP_RERUN_JSON, 'r', encoding='utf-8') as f:
        rerun_records = json.load(f)
    for r in rerun_records:
        base_map[r['keyword']] = r
    print(f'재추출 결과 {len(rerun_records)}개 → override 완료')
else:
    print('IP_속성_재추출.json 없음 → 기존 결과만 사용')

merged_ip = list(base_map.values())
with open(IP_MERGED_JSON, 'w', encoding='utf-8') as f:
    json.dump(merged_ip, f, ensure_ascii=False, indent=2)
print(f'병합 완료: {IP_MERGED_JSON} ({len(merged_ip)}개 IP)')

기존 IP_속성.json: 255개
재추출 결과 72개 → override 완료
병합 완료: C:\Users\송정현\Documents\Projects\박재홍교수님세미나\Projects\20기\7eleven_npd_framework\data\processed\IP_속성추출\IP_속성_통합.json (291개 IP)


In [20]:
# [수동 검수용 — 필요 시 주석 해제]
# # ── Step 3. signature_keywords + 확정된키워드(Sheet2) 합산 export ──
# # 소스1: IP_속성_통합.json → signature_keywords
# # 소스2: keyword_vocab_review_final.xlsx [IP 시트] → 확정된 키워드
# from collections import Counter, defaultdict
#
# VOCAB_FINAL_PATH = os.path.join(BASE_DIR, 'data', 'processed', 'keyword_vocab_review_final.xlsx')
# IP_KW_VOCAB_PATH = os.path.join(BASE_DIR, 'data', 'processed', 'ip_keyword_vocab_review.xlsx')
#
# def _ip_safe_list(val):
#     if isinstance(val, list): return val
#     if isinstance(val, str):
#         try: return ast.literal_eval(val)
#         except: return [v.strip() for v in val.split(',') if v.strip()]
#     return []
#
# # ── 소스1: IP_속성_통합.json ─────────────────────────────────────
# kw_src1 = defaultdict(list)
# for r in merged_ip:
#     ip_name = r.get('keyword', '')
#     for kw in _ip_safe_list(r.get('signature_keywords', [])):
#         if isinstance(kw, str) and kw.strip():
#             kw_src1[kw.strip()].append(ip_name)
# print(f'소스1 (재추출): 고유 키워드 {len(kw_src1)}개')
#
# # ── 소스2: keyword_vocab_review_final.xlsx [IP 시트] ─────────────
# kw_src2 = defaultdict(list)
# if os.path.exists(VOCAB_FINAL_PATH):
#     df_ip_sheet = pd.read_excel(VOCAB_FINAL_PATH, sheet_name='IP')
#     for _, row in df_ip_sheet.iterrows():
#         ip_name = str(row.get('IP명', '')).strip()
#         raw = row.get('확정된 키워드', '')
#         if pd.isna(raw) or str(raw).strip() == '':
#             continue
#         for kw in [v.strip() for v in str(raw).split(',') if v.strip()]:
#             kw_src2[kw].append(ip_name)
#     print(f'소스2 (확정본): 고유 키워드 {len(kw_src2)}개')
# else:
#     print(f'⚠ {VOCAB_FINAL_PATH} 없음 → 소스2 건너뜀')
#
# # ── 합산 ─────────────────────────────────────────────────────────
# all_kws = set(kw_src1) | set(kw_src2)
# print(f'전체 고유 키워드: {len(all_kws)}개')
#
# def _flag_compound(kw):
#     if ' ' in kw:
#         return '공백포함'
#     if len(kw) >= 5 and not re.search(r'[a-zA-Z0-9]', kw):
#         return '길이확인'
#     return ''
#
# def _source_label(kw):
#     in1, in2 = kw in kw_src1, kw in kw_src2
#     if in1 and in2: return '공통'
#     if in1: return '재추출'
#     return '확정본'
#
# rows = []
# for kw in sorted(all_kws):
#     ips = list(dict.fromkeys(kw_src1.get(kw, []) + kw_src2.get(kw, [])))
#     rows.append({
#         '키워드':       kw,
#         '빈도':         len(ips),
#         '소스':         _source_label(kw),
#         '출현IP':       ', '.join(ips[:5]),
#         '복합어플래그': _flag_compound(kw),
#         '제거/검토':    '',
#     })
#
# df_ip_vocab = pd.DataFrame(rows).sort_values(['소스', '빈도'], ascending=[True, False])
# df_ip_vocab.to_excel(IP_KW_VOCAB_PATH, index=False)
#
# print(f'저장: {IP_KW_VOCAB_PATH}')
# print(f'  전체       : {len(df_ip_vocab)}개')
# print(f'  공통       : {(df_ip_vocab["소스"]=="공통").sum()}개')
# print(f'  재추출 전용: {(df_ip_vocab["소스"]=="재추출").sum()}개')
# print(f'  확정본 전용: {(df_ip_vocab["소스"]=="확정본").sum()}개')
# print(f'  공백포함   : {(df_ip_vocab["복합어플래그"]=="공백포함").sum()}개')
# print(f'  길이확인   : {(df_ip_vocab["복합어플래그"]=="길이확인").sum()}개')


In [21]:
# ── Step 4. ip_keyword_vocab_review_final.xlsx → IP_키워드사전.json ──
# 규칙:
#   제거/검토 = 'O'          → 키워드 제거
#   제거/검토 = 빈칸          → 원래 키워드 유지
#   제거/검토 = '단어A,단어B' → 원래 키워드를 단어A, 단어B로 교체
#   vocab에 없는 키워드       → 그대로 유지 (안전 fallback)

import os, json
import pandas as pd

IP_VOCAB_FINAL_PATH = os.path.join(BASE_DIR, 'data', 'processed', 'ip_keyword_vocab_review_final.xlsx')
IP_KW_DICT_PATH     = os.path.join(BASE_DIR, 'data', 'processed', 'IP_속성추출', 'IP_키워드사전.json')
IP_MERGED_JSON      = os.path.join(BASE_DIR, 'data', 'processed', 'IP_속성추출', 'IP_속성_통합.json')

# ── 교정 맵 구축 ──
df_vocab = pd.read_excel(IP_VOCAB_FINAL_PATH)
correction_map = {}
for _, row in df_vocab.iterrows():
    orig   = str(row['키워드']).strip()
    review = row.get('제거/검토', '')
    if pd.isna(review) or str(review).strip() == '':
        correction_map[orig] = [orig]
    elif str(review).strip() == 'O':
        correction_map[orig] = []
    else:
        correction_map[orig] = [k.strip() for k in str(review).split(',') if k.strip()]

print(f'교정 맵: 총 {len(correction_map)}개 키워드')
print(f'  제거: {sum(1 for v in correction_map.values() if not v)}개')
print(f'  교체: {sum(1 for k, v in correction_map.items() if v and v != [k])}개')
print(f'  유지: {sum(1 for k, v in correction_map.items() if v == [k])}개')

# ── IP_속성_통합.json 로드 ──
with open(IP_MERGED_JSON, 'r', encoding='utf-8') as f:
    merged_data = json.load(f)

# ── IP별 키워드 교정 적용 ──
result = []
for ip in merged_data:
    original_kws = ip.get('signature_keywords', [])
    final_kws = []
    for kw in original_kws:
        corrected = correction_map.get(kw, [kw])
        final_kws.extend(corrected)
    final_kws = list(dict.fromkeys(final_kws))  # 중복 제거 (순서 유지)

    result.append({
        'ip_name':   ip['keyword'],
        'ip_type':   ip.get('ip_type', ''),
        'target_age': ip.get('target_age', []),
        '최종키워드': final_kws,
    })

# ── 저장 ──
with open(IP_KW_DICT_PATH, 'w', encoding='utf-8') as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

print(f'IP_키워드사전.json 저장 완료: {IP_KW_DICT_PATH}')
print(f'총 {len(result)}개 IP')
empties = [r['ip_name'] for r in result if not r['최종키워드']]
if empties:
    print(f'⚠ 최종키워드 비어있는 IP: {empties}')
else:
    print('✓ 모든 IP 키워드 정상')



교정 맵: 총 920개 키워드
  제거: 115개
  교체: 234개
  유지: 571개
IP_키워드사전.json 저장 완료: C:\Users\송정현\Documents\Projects\박재홍교수님세미나\Projects\20기\7eleven_npd_framework\data\processed\IP_속성추출\IP_키워드사전.json
총 291개 IP
⚠ 최종키워드 비어있는 IP: ['델토리', '스스스', '정성진', '최재승']


### Step 4.5. IP 수동입력 병합

**목적**: `IP_속성_수동입력.json` (재추출 불가 IP 수동 검토분)을 `IP_속성.json` 및 `IP_키워드사전.json`에 병합  
**입력**: `IP_속성추출/IP_속성_수동입력.json`  
**출력**: `IP_속성.json` 업데이트 + `IP_키워드사전.json` 패치 (Step 4 출력 포맷 유지)

In [22]:
import json, os

_IP_MANUAL = os.path.join(BASE_DIR, 'data', 'processed', 'IP_속성추출', 'IP_속성_수동입력.json')
_IP_ATTR   = os.path.join(BASE_DIR, 'data', 'processed', 'IP_속성추출', 'IP_속성.json')
_IP_KW_DICT = os.path.join(BASE_DIR, 'data', 'processed', 'IP_속성추출', 'IP_키워드사전.json')

if not os.path.exists(_IP_MANUAL):
    print('IP_속성_수동입력.json 없음 → skip')
else:
    # ── IP_속성.json 업데이트 ──────────────────────────────────────────
    with open(_IP_ATTR, 'r', encoding='utf-8') as f:
        _existing = json.load(f)
    with open(_IP_MANUAL, 'r', encoding='utf-8') as f:
        _manual = json.load(f)

    _existing_kws = {r['keyword'] for r in _existing}
    _added = [r for r in _manual if r['keyword'] not in _existing_kws]
    _merged_attr = _existing + _added

    with open(_IP_ATTR, 'w', encoding='utf-8') as f:
        json.dump(_merged_attr, f, ensure_ascii=False, indent=2)
    print(f'IP_속성.json: 기존 {len(_existing)}개 + 신규 {len(_added)}개 = {len(_merged_attr)}개')

    # ── IP_키워드사전.json 패치 (Step 4 포맷 유지: ip_name / 최종키워드) ──
    with open(_IP_KW_DICT, 'r', encoding='utf-8') as f:
        _kw_dict = json.load(f)

    _existing_ip_names = {e['ip_name'] for e in _kw_dict}
    _patch = [
        {
            'ip_name':    r['keyword'],
            'ip_type':    r.get('ip_type', ''),
            'target_age': r.get('target_age', []),
            '최종키워드':  r.get('signature_keywords', []),
        }
        for r in _added
        if r.get('signature_keywords')
    ]
    _kw_dict.extend(_patch)

    with open(_IP_KW_DICT, 'w', encoding='utf-8') as f:
        json.dump(_kw_dict, f, ensure_ascii=False, indent=2)
    print(f'IP_키워드사전.json: +{len(_patch)}개 추가 → 총 {len(_kw_dict)}개')
    if _patch:
        print('  추가된 IP:', [p['ip_name'] for p in _patch])


IP_속성.json: 기존 255개 + 신규 0개 = 255개
IP_키워드사전.json: +0개 추가 → 총 291개


## Phase 4. 수동 검수 반영 & 최종 키워드 완성

`keyword_vocab_review_final.xlsx` (수동 검수 완료본)을 3소스 parquet에 패치 후 B4와 조인하여 최종 저장
- PROMO_MAP: '프로모션' 태그 → 표준 프로모션 코드(0106/0107/0205) 변환
- **출력**: `final_product_keywords.parquet`

In [23]:
# ── PROMO_MAP: '프로모션' 태그 키워드 → 표준 프로모션 코드 매핑 ──
# keyword_vocab_review_final.xlsx 에서 제거/검토 = '프로모션'인 키워드를
# 01_instagram_data_normalization_and_cleaning.ipynb 의 categorized_dict 체계와 동일하게 정규화

PROMO_MAP = {
    # 0106 : 단품할인
    '단독할인':    '0106 : 단품할인',
    '특가':        '0106 : 단품할인',
    '특별할인':    '0106 : 단품할인',
    '할인판매':    '0106 : 단품할인',
    '기간한정':    '0106 : 단품할인',
    '한정상품':    '0106 : 단품할인',
    '한정수량':    '0106 : 단품할인',
    '한정이벤트':  '0106 : 단품할인',
    '한정판매':    '0106 : 단품할인',
    '사전공개':    '0106 : 단품할인',
    '앵콜전':      '0106 : 단품할인',
    '오픈런':      '0106 : 단품할인',
    '오프라인판매': '0106 : 단품할인',
    '구매이벤트':  '0106 : 단품할인',
    '히든이벤트':  '0106 : 단품할인',
    # 0107 : 묶음할인(구간)
    '번들':        '0107 : 묶음할인(구간)',
    '기획전':      '0107 : 묶음할인(구간)',
    '기획패키지':  '0107 : 묶음할인(구간)',
    '한정구성':    '0107 : 묶음할인(구간)',
    '한정패키지':  '0107 : 묶음할인(구간)',
    '사은품증정':  '0107 : 묶음할인(구간)',
    '선착순증정':  '0107 : 묶음할인(구간)',
    '쟁여두기':    '0107 : 묶음할인(구간)',
    # 0205 : 장바구니할인
    '구매시':      '0205 : 장바구니할인',
    '구매조건':    '0205 : 장바구니할인',
    '구매혜택':    '0205 : 장바구니할인',
    '리워드제공':  '0205 : 장바구니할인',
    '포인트적립':  '0205 : 장바구니할인',
    '신한SOL페이': '0205 : 장바구니할인',
    '토스페이':    '0205 : 장바구니할인',
}
print(f'PROMO_MAP 로드: {len(PROMO_MAP)}개 키워드')


PROMO_MAP 로드: 30개 키워드


In [24]:
# ── [수동 검수 반영] keyword_vocab_review_final → 3소스 parquet 패치 ──
# 규칙:
#   제거/검토 열 = 'O'          → 키워드 완전 제거
#   제거/검토 열 = '프로모션'   → PROMO_MAP 조회 → 표준 프로모션 코드로 교체
#   제거/검토 열 = '가공, 버터' → 원 키워드를 해당 키워드들로 교체 (1→N 분리 포함)
#   제거/검토 열 = 빈 값        → 현행 유지
#
# ※ VOCAB_REVIEW_PATH(원본 export)와 충돌 방지를 위해 별도 경로 사용

VOCAB_FINAL_PATH = os.path.join(BASE_DIR, 'data', 'processed', 'keyword_vocab_review_final.xlsx')
REVIEW_COL = '제거/검토'

df_vr = pd.read_excel(VOCAB_FINAL_PATH)
if REVIEW_COL not in df_vr.columns:
    raise ValueError(f"열 '{REVIEW_COL}' 없음: {list(df_vr.columns)}")
print(f'검수 파일 로드: {VOCAB_FINAL_PATH}')
print(f'총 {len(df_vr)}개 키워드')

remove_set  = set()
replace_map = {}

for _, row in df_vr.iterrows():
    kw  = str(row['키워드']).strip()
    val = row.get(REVIEW_COL, '')
    if pd.isna(val) or str(val).strip() == '':
        continue
    val = str(val).strip()
    if val == 'O':
        remove_set.add(kw)
    elif val == '프로모션':
        promo_code = PROMO_MAP.get(kw)
        if promo_code:
            replace_map[kw] = [promo_code]
        else:
            remove_set.add(kw)
    else:
        replacements = [r.strip() for r in val.split(',') if r.strip()]
        if replacements:
            replace_map[kw] = replacements

promo_count = sum(1 for v in replace_map.values() if len(v) == 1 and v[0].startswith('0'))
print(f'교정 맵 구축:')
print(f'  제거 (O)        : {len(remove_set):,}개')
print(f'  프로모션 코드화 : {promo_count}개')
print(f'  대체 키워드     : {len(replace_map):,}개  (1→N 분리 포함)')

def _safe_list(val):
    if isinstance(val, list): return val
    try: return list(val)
    except TypeError: pass
    if isinstance(val, str):
        try: return ast.literal_eval(val)
        except: return []
    return []

def apply_vocab_patch(kw_list):
    result = []
    for kw in _safe_list(kw_list):
        if not isinstance(kw, str) or not kw:
            continue
        if kw in remove_set:
            continue
        elif kw in replace_map:
            result.extend(replace_map[kw])
        else:
            result.append(kw)
    return list(dict.fromkeys(result))

patch_targets = [
    (INSTA_PROC_PATH, '인스타',  ['확정키워드_정제']),
    (BLOG_PROC_PATH,  '블로그',  ['확정키워드_정제']),
    (TREND_PROC_PATH, '트렌드',  ['확정키워드_정제', 'trend_flag_kws']),
]

for path, name, cols in patch_targets:
    df_ = pd.read_parquet(path)
    before = df_['확정키워드_정제'].apply(lambda x: len(_safe_list(x))).sum()
    for col in cols:
        if col in df_.columns:
            df_[col] = df_[col].apply(apply_vocab_patch)
    after = df_['확정키워드_정제'].apply(len).sum()
    df_.to_parquet(path, index=False, engine='pyarrow')
    print(f'  {name}: {before:,} → {after:,}개  (Δ{after - before:+,d})')

print('\n교정 완료 → join-all (Block 5) 이어서 실행하세요')


검수 파일 로드: C:\Users\송정현\Documents\Projects\박재홍교수님세미나\Projects\20기\7eleven_npd_framework\data\processed\keyword_vocab_review_final.xlsx
총 3463개 키워드
교정 맵 구축:
  제거 (O)        : 71개
  프로모션 코드화 : 30개
  대체 키워드     : 1,807개  (1→N 분리 포함)
  인스타: 23,727 → 23,783개  (Δ+56)
  블로그: 15,499 → 16,663개  (Δ+1,164)
  트렌드: 38,798 → 37,740개  (Δ-1,058)

교정 완료 → join-all (Block 5) 이어서 실행하세요


### Phase 4B. 보정 후 키워드 검토 Export *(수동 검수용 — 재검수 시 활성화)*

> 최초 1회 실행 후 수동 검수 완료. 재검수가 필요할 때만 아래 셀을 활성화.


In [25]:
# [수동 검수용 — 필요 시 주석 해제]
# # ── Phase 4B: 보정 후 최종 키워드 검토 export ─────────────────────
# # keyword_vocab_review_final.xlsx 와 동일한 형식
# # 차이: 보정 적용 후 parquet 기준 → 실제 최종 키워드 풀 반영
# FINAL_KW_REVIEW_PATH = os.path.join(BASE_DIR, 'data', 'processed', 'final_keyword_review.xlsx')
#
# def _to_list(val):
#     if isinstance(val, list): return val
#     try: return list(val)
#     except TypeError: pass
#     if isinstance(val, str):
#         try: return ast.literal_eval(val)
#         except: return []
#     return []
#
# df_i_rev = pd.read_parquet(INSTA_PROC_PATH)
# df_b_rev = pd.read_parquet(BLOG_PROC_PATH)
# df_t_rev = pd.read_parquet(TREND_PROC_PATH)
#
# from collections import Counter
# def _count(df, col='확정키워드_정제'):
#     return Counter(kw for lst in df[col].apply(_to_list) for kw in lst if kw)
#
# cnt_i = _count(df_i_rev)
# cnt_b = _count(df_b_rev)
# cnt_t = _count(df_t_rev)
# all_kws = set(cnt_i) | set(cnt_b) | set(cnt_t)
#
# rows = []
# for kw in all_kws:
#     fi, fb, ft = cnt_i.get(kw, 0), cnt_b.get(kw, 0), cnt_t.get(kw, 0)
#     rows.append({
#         '키워드':     kw,
#         '총빈도':     fi + fb + ft,
#         '인스타':     '✓' if fi > 0 else '',
#         '블로그':     '✓' if fb > 0 else '',
#         '트렌드':     '✓' if ft > 0 else '',
#         '인스타빈도': fi,
#         '블로그빈도': fb,
#         '트렌드빈도': ft,
#         '제거/검토':  '',
#         '브랜드/IP':  '',
#     })
#
# df_out = (pd.DataFrame(rows)
#           .sort_values('총빈도', ascending=False)
#           .reset_index(drop=True))
#
# df_out.to_excel(FINAL_KW_REVIEW_PATH, index=False)
#
# print(f'저장: {FINAL_KW_REVIEW_PATH}')
# print(f'  전체 고유 키워드: {len(df_out):,}개')
# print(f'  인스타 포함: {(df_out["인스타"]=="✓").sum():,}개')
# print(f'  블로그 포함: {(df_out["블로그"]=="✓").sum():,}개')
# print(f'  트렌드 포함: {(df_out["트렌드"]=="✓").sum():,}개')
# print(f'  3소스 공통:  {((df_out["인스타"]=="✓") & (df_out["블로그"]=="✓") & (df_out["트렌드"]=="✓")).sum():,}개')


### Phase 4C. 2차 키워드 보정 반영

**입력**: `data/processed/final_keyword_review_fix.xlsx`  
**규칙**:  
- `제거/검토` = `O` → 해당 키워드 제거  
- `제거/검토` = `가공, 버터` → 원 키워드를 `가공`, `버터`로 분리  
- `제거/검토` = `가챠` → 원 키워드를 `가챠`로 교체  
- 비어 있음 → 현행 유지

In [26]:
# ── Phase 4C: final_keyword_review_fix.xlsx → 3소스 parquet 2차 패치 ──
FINAL_FIX_PATH = os.path.join(BASE_DIR, 'data', 'processed', 'final_keyword_review_fix.xlsx')
REVIEW_COL = '제거/검토'

df_fix = pd.read_excel(FINAL_FIX_PATH)
print(f'2차 보정 파일 로드: {len(df_fix)}개 키워드')

remove_set  = set()
replace_map = {}

for _, row in df_fix.iterrows():
    kw  = str(row['키워드']).strip()
    val = row.get(REVIEW_COL, '')
    if pd.isna(val) or str(val).strip() == '':
        continue
    val = str(val).strip()

    if val == 'O':
        remove_set.add(kw)
    else:
        # 따옴표로 감싸진 경우 ('0107 : 묶음할인(구간)') → 따옴표 제거
        if val.startswith("'") and val.endswith("'"):
            val = val[1:-1].strip()
        replacements = [r.strip() for r in val.split(',') if r.strip()]
        if replacements:
            replace_map[kw] = replacements

print(f'  제거 (O)    : {len(remove_set):,}개')
print(f'  교체/분리   : {len(replace_map):,}개')

def _safe_list(val):
    if isinstance(val, list): return val
    try: return list(val)
    except TypeError: pass
    if isinstance(val, str):
        try: return ast.literal_eval(val)
        except: return []
    return []

def apply_fix_patch(kw_list):
    result = []
    for kw in _safe_list(kw_list):
        if not isinstance(kw, str) or not kw:
            continue
        if kw in remove_set:
            continue
        elif kw in replace_map:
            result.extend(replace_map[kw])
        else:
            result.append(kw)
    return list(dict.fromkeys(result))

patch_targets = [
    (INSTA_PROC_PATH, '인스타',  ['확정키워드_정제']),
    (BLOG_PROC_PATH,  '블로그',  ['확정키워드_정제']),
    (TREND_PROC_PATH, '트렌드',  ['확정키워드_정제', 'trend_flag_kws']),
]

for path, name, cols in patch_targets:
    df_ = pd.read_parquet(path)
    before = df_['확정키워드_정제'].apply(lambda x: len(_safe_list(x))).sum()
    for col in cols:
        if col in df_.columns:
            df_[col] = df_[col].apply(apply_fix_patch)
    after = df_['확정키워드_정제'].apply(len).sum()
    df_.to_parquet(path, index=False, engine='pyarrow')
    print(f'  {name}: {before:,} → {after:,}개  (Δ{after - before:+,d})')

print('2차 보정 완료')


2차 보정 파일 로드: 1878개 키워드
  제거 (O)    : 16개
  교체/분리   : 71개
  인스타: 23,783 → 23,781개  (Δ-2)
  블로그: 16,663 → 16,624개  (Δ-39)
  트렌드: 37,740 → 37,703개  (Δ-37)
2차 보정 완료


### Phase 2D. 편의점 인스타그램 제품 언급 데이터 추출

**입력**:  +  (세븐일레븐·CU·GS25)  
**출력**:   
  - 스키마:  /  /  /  /  /  /   
  - 세븐일레븐:  = review_final 보정값 / CU·GS25:  = None (미검토)  
  -  행은 delete_set으로 처리하여 출력에서 제외

In [27]:
# ── Phase 2D: 편의점 인스타그램 제품 언급 데이터 추출 ────────────────
# 입력: product_name_review_final.xlsx (검토 완료본)
#       편의점_인스타/{단일,다중}/*.xlsx (세븐·CU·GS25)
# 출력: data/processed/product_instagram_engagement.parquet (.parquet + .csv)
# 스키마: 편의점(세븐일레븐/CU/GS25) | 원본명 | 정규화명(미검토=None) | 좋아요 수 | 언급일 | url | body
#
# 수정명 컬럼 해석:
#   공백/NaN  → 정규화명 그대로 유지
#   'O'       → delete_set (해당 제품 전체 제거)
#   그 외     → rename_map[원본명] = 수정명 (덮어쓰기)

import os
import re
import pandas as pd

BASE_DIR          = r'C:\Users\송정현\Documents\Projects\박재홍교수님세미나\Projects\20기\7eleven_npd_framework'
NAME_REVIEW_PATH  = os.path.join(BASE_DIR, 'data', 'processed', 'product_name_review_final.xlsx')
OUT_PATH          = os.path.join(BASE_DIR, 'data', 'processed', 'product_instagram_engagement.parquet')
OUT_CSV           = os.path.join(BASE_DIR, 'data', 'processed', 'product_instagram_engagement.csv')
INSTA_DIR         = os.path.join(BASE_DIR, 'data', '편의점', '편의점_인스타')

# 소스 정의: (브랜드, 단일파일, 다중파일, 제외플래그컬럼, 제외값)
SOURCES = [
    ('세븐일레븐',
     os.path.join(INSTA_DIR, '단일', '세븐_단일.xlsx'),
     os.path.join(INSTA_DIR, '다중', '세븐_다중.xlsx'),
     '제외여부', 'o'),
    ('CU',
     os.path.join(INSTA_DIR, '단일', 'CU단일.xlsx'),
     os.path.join(INSTA_DIR, '다중', 'CU다중.xlsx'),
     '크롤링 상태', '제외'),
    ('GS25',
     os.path.join(INSTA_DIR, '단일', 'GS25_단일.xlsx'),
     os.path.join(INSTA_DIR, '다중', 'GS25_다중.xlsx'),
     'Notes', 'X'),
]

# ── 1. product_name_review_final.xlsx → (delete_set, rename_map) ──
def load_name_patch(path: str) -> tuple[set, dict]:
    """
    수정명 해석:
      NaN / ''  → 정규화명 유지
      'O'       → 제거 대상 (delete_set)
      기타      → 해당 값으로 대체 (rename_map)
    원본 parquet은 건드리지 않고, 읽는 시점에 패치를 적용하는 방식.
    """
    df = pd.read_excel(path)
    delete_set, rename_map = set(), {}
    for _, row in df.iterrows():
        원본명  = str(row['원본명']).strip()
        정규화명 = str(row['정규화명']).strip()
        수정명  = str(row.get('수정명', '')).strip() if pd.notna(row.get('수정명')) else ''
        if 수정명.upper() == 'O':
            delete_set.add(원본명)
        elif 수정명:
            rename_map[원본명] = 수정명
        else:
            rename_map[원본명] = 정규화명
    return delete_set, rename_map

delete_set, rename_map = load_name_patch(NAME_REVIEW_PATH)
print(f'패치 로드 완료 — 제거 대상: {len(delete_set):,}개 / 리네임: {len(rename_map):,}개')

# ── 2. formatted_output 파싱: ["제품명", 가격, 용량] 패턴 ───────────
def parse_product_names(fo_str: str) -> list:
    """formatted_output에서 제품명(리스트 첫 원소) 전체 추출"""
    if not isinstance(fo_str, str):
        return []
    return re.findall(r"""\[["'](.+?)["'],\s*\d+""", fo_str)

# ── 3. 파일 단위 처리 ──────────────────────────────────────────────
def process_file(path: str, brand: str, excl_col: str, excl_val: str) -> list:
    if not os.path.exists(path):
        print(f'  [SKIP] 파일 없음: {os.path.basename(path)}')
        return []

    df = pd.read_excel(path)
    total = len(df)

    # 제외 플래그 필터
    if excl_col in df.columns:
        df = df[df[excl_col].astype(str).str.strip() != excl_val]

    print(f'  [{brand}] {os.path.basename(path)}: {total:,}행 → {len(df):,}행 (제외 {total - len(df):,}건)')

    rows = []
    for _, row in df.iterrows():
        products = parse_product_names(str(row.get('formatted_output', '')))
        for prod in products:
            if prod in delete_set:          # 제거 대상 스킵
                continue
            rows.append({
                '편의점':   brand,
                '원본명':   prod,
                '정규화명': rename_map.get(prod, None),  # 미검토(CU/GS25) → None
                '좋아요 수': row.get('likes', 0),
                '언급일':   row.get('date', ''),
                'url':      row.get('url', ''),
                'body':     row.get('body', ''),
            })
    return rows

# ── 4. 전체 소스 순회 ──────────────────────────────────────────────
all_rows = []
for brand, path_단일, path_다중, excl_col, excl_val in SOURCES:
    print(f'\n[{brand}]')
    all_rows += process_file(path_단일, brand, excl_col, excl_val)
    all_rows += process_file(path_다중, brand, excl_col, excl_val)

# ── 5. 저장 ──────────────────────────────────────────────────────
COLS = ['편의점', '원본명', '정규화명', '좋아요 수', '언급일', 'url', 'body']
# 편의점 값: '세븐일레븐' / 'CU' / 'GS25'
# 정규화명:  세븐일레븐 제품 → rename_map 보정값 / CU·GS25 → None (미검토)
result = pd.DataFrame(all_rows, columns=COLS)
result['언급일'] = pd.to_datetime(result['언급일'], errors='coerce')
result['좋아요 수'] = pd.to_numeric(result['좋아요 수'], errors='coerce').fillna(0).astype(int)
result = result.sort_values(['정규화명', '언급일']).reset_index(drop=True)

result.to_parquet(OUT_PATH, index=False, engine='pyarrow')
result.to_csv(OUT_CSV, index=False, encoding='utf-8-sig')

print(f'\n=== Phase 2D 완료: product_instagram_engagement.parquet ===')
print(f'  전체 행 수            : {len(result):,}행')
print(f'  편의점별:\n{result["편의점"].value_counts().to_string()}')
print(f'  고유 원본명           : {result["원본명"].nunique():,}개')
print(f'  고유 정규화명         : {result["정규화명"].nunique():,}개')
print(f'  정규화명 있음(세븐)   : {result["정규화명"].notna().sum():,}건')
print(f'  정규화명 None(미검토) : {result["정규화명"].isna().sum():,}건')
print(f'  언급일 범위           : {result["언급일"].min().date()} ~ {result["언급일"].max().date()}')
result.head(5)

패치 로드 완료 — 제거 대상: 226개 / 리네임: 4,388개

[세븐일레븐]
  [세븐일레븐] 세븐_단일.xlsx: 291행 → 238행 (제외 53건)
  [세븐일레븐] 세븐_다중.xlsx: 1,342행 → 1,085행 (제외 257건)

[CU]
  [CU] CU단일.xlsx: 505행 → 505행 (제외 0건)
  [CU] CU다중.xlsx: 1,901행 → 1,901행 (제외 0건)

[GS25]
  [GS25] GS25_단일.xlsx: 560행 → 533행 (제외 27건)
  [GS25] GS25_다중.xlsx: 1,336행 → 1,336행 (제외 0건)

=== Phase 2D 완료: product_instagram_engagement.parquet ===
  전체 행 수            : 4,972행
  편의점별:
편의점
GS25     1868
CU       1803
세븐일레븐    1301
  고유 원본명           : 3,858개
  고유 정규화명         : 678개
  정규화명 있음(세븐)   : 1,272건
  정규화명 None(미검토) : 3,700건
  언급일 범위           : 2025-01-01 ~ 2025-12-31


,편의점,원본명,정규화명,좋아요 수,언급일,url,body
0,세븐일레븐,25년 꿀고구마호빵,25년꿀고구마호빵,2419,2025-10-14,https://www.instagram.com/p/DPzqtjoEnBk/,☁️찬 바람 불 땐? 무조건 호빵이지!\n입에 넣자마자 사르르 녹는 호빵이\n4가지...
1,세븐일레븐,25년 듬뿍피자호빵,25년듬뿍피자호빵,2419,2025-10-14,https://www.instagram.com/p/DPzqtjoEnBk/,☁️찬 바람 불 땐? 무조건 호빵이지!\n입에 넣자마자 사르르 녹는 호빵이\n4가지...
2,세븐일레븐,25년 생생야채호빵,25년생생야채호빵,2419,2025-10-14,https://www.instagram.com/p/DPzqtjoEnBk/,☁️찬 바람 불 땐? 무조건 호빵이지!\n입에 넣자마자 사르르 녹는 호빵이\n4가지...
3,세븐일레븐,25년 정통단팥호빵,25년흑당단팥호빵,2419,2025-10-14,https://www.instagram.com/p/DPzqtjoEnBk/,☁️찬 바람 불 땐? 무조건 호빵이지!\n입에 넣자마자 사르르 녹는 호빵이\n4가지...
4,세븐일레븐,빚은찹쌀떡,25수능 빚은찹쌀떡,1071,2025-11-11,https://www.instagram.com/p/DQ5npqhAcK8/,행운의 기운 받아랏 얍! 🍀🍀🍀\n지금도 열심히 공부하고 있을 \n소중한 친구를 위...


## Phase 4D. Engagement + POS-Blog 키워드 데이터셋 빌드

**DS1** `instagram_engagement_with_keywords.parquet`  
스키마: `편의점명 | 원본명 | 정규화명 | 키워드 | 좋아요 수 | 언급일 | url | body`

**DS2** `pos_blog_keywords_bridge.parquet`  
스키마: `원본명(POS명) | 정규화명 | 키워드(블로그 기반)`  
- NPD 상품만 (is_npd=True), 정규화명 기준 블로그 키워드 union


In [28]:
# ── Phase 4D: DS1 — Instagram Engagement + 키워드 ────────────────
import polars as _pl

_ENG_PATH    = os.path.join(BASE_DIR, 'data', 'processed', 'product_instagram_engagement.csv')
_INSTA_PATH  = INSTA_PROC_PATH   # insta_keywords_processed.parquet
_OUT_DS1     = os.path.join(BASE_DIR, 'data', 'processed', 'instagram_engagement_with_keywords.parquet')

In [29]:
# Phase 4D-1: CU/GS 검수 라벨 부착
_EXCEL_DIR = os.path.join(BASE_DIR, 'data', 'processed', '편의점_instagram')
_CU_XL = os.path.join(_EXCEL_DIR, 'CU_instagram_completed.xlsx')
_GS_XL = os.path.join(_EXCEL_DIR, 'GS_instagram_completed.xlsx')

_cu_lbl = pd.read_excel(_CU_XL)
_gs_lbl = pd.read_excel(_GS_XL)

_BASE_COLS = ['편의점', '원본명', '정규화명', '좋아요 수', '언급일', 'url', 'body']
_REVIEW_COLS = [c for c in _cu_lbl.columns if c not in _BASE_COLS]

for _xl in [_cu_lbl, _gs_lbl]:
    _xl['_key'] = _xl['url'] + '||' + _xl['원본명'].astype(str)

_label_map = pd.concat([
    _cu_lbl.drop_duplicates('_key').set_index('_key')[_REVIEW_COLS],
    _gs_lbl.drop_duplicates('_key').set_index('_key')[_REVIEW_COLS],
])

_ds1 = pd.read_parquet(_OUT_DS1)
_ds1 = _ds1.drop(columns=[c for c in _ds1.columns if c.startswith('제외') or c.startswith('다시 봐야할 거')], errors='ignore')
_ds1['_key'] = _ds1['url'] + '||' + _ds1['원본명'].astype(str)
_ds1 = _ds1.merge(_label_map.reset_index(), on='_key', how='left').drop(columns=['_key'])
_ds1.to_parquet(_OUT_DS1, index=False, engine='pyarrow')

print(f'[검수 라벨 부착] 컬럼: {_REVIEW_COLS}')
_cu_excl = _ds1[_ds1['편의점명']=='CU']['제외'].notna().sum()
_gs_excl = _ds1[_ds1['편의점명']=='GS25']['제외'].notna().sum()
print(f'  CU 제외 표기: {_cu_excl}건 / GS 제외 표기: {_gs_excl}건')
del _cu_lbl, _gs_lbl, _label_map


[검수 라벨 부착] 컬럼: ['제외', '다시 봐야할 거']
  CU 제외 표기: 1건 / GS 제외 표기: 0건


In [30]:
# Phase 4D-2: CU/GS/세븐일레븐 IP_NM 컬럼 추가
# Cell 46 (Phase 4D-1) 바로 다음, _EXCEL_DIR / _OUT_DS1 변수 재사용

_BRIDGE_PATH = os.path.join(BASE_DIR, 'data', 'processed', 'seven_eleven_product_master.parquet')
_CU_IP_XL  = os.path.join(_EXCEL_DIR, 'CU_instagram_ip.xlsx')
_GS_IP_XL  = os.path.join(_EXCEL_DIR, 'GS_instagram_IP.xlsx')
_SEVEN_XL  = os.path.join(_EXCEL_DIR, 'Seven_instagram_completed.xlsx')

def _norm_id(x):
    try: return str(int(float(x)))
    except: return str(x)

# ── CU / GS: url+원본명 키 직접 매핑 ───────────────────────────────
_cu_ip = pd.read_excel(_CU_IP_XL).rename(columns={'ip': 'IP_NM'})
_gs_ip = pd.read_excel(_GS_IP_XL)
_gs_ip['편의점'] = _gs_ip['편의점'].fillna('GS25')
_gs_ip = _gs_ip.rename(columns={'IP': 'IP_NM'})

for _df in [_cu_ip, _gs_ip]:
    _df['_key'] = _df['url'].fillna('') + '||' + _df['원본명'].astype(str)

_cu_gs_map = pd.concat([
    _cu_ip[_cu_ip['IP_NM'].notna()].drop_duplicates('_key').set_index('_key')[['IP_NM']],
    _gs_ip[_gs_ip['IP_NM'].notna()].drop_duplicates('_key').set_index('_key')[['IP_NM']],
])

# ── 세븐일레븐: ITEM_CD → 브릿지 → 인스타_정규화명 경유 ────────────
_seven = pd.read_excel(_SEVEN_XL)
_seven = _seven[_seven['IP'].notna()][['ITEM_CD', 'IP']].copy()
_seven['ITEM_CD'] = _seven['ITEM_CD'].apply(_norm_id)

_bridge = pd.read_parquet(_BRIDGE_PATH)
_bridge['ITEM_CD'] = _bridge['ITEM_CD'].apply(_norm_id)
_bridge = _bridge[_bridge['인스타_정규화명'].notna()][['ITEM_CD', '인스타_정규화명']]

_seven_map = (
    _seven.merge(_bridge, on='ITEM_CD', how='inner')
    .drop_duplicates('인스타_정규화명')
    .set_index('인스타_정규화명')['IP']
    .rename('IP_NM')
)

# ── parquet 로드 → IP_NM 채우기 ────────────────────────────────────
_ds1 = pd.read_parquet(_OUT_DS1)
_ds1['IP_NM'] = None

# CU / GS
_ds1['_key'] = _ds1['url'].fillna('') + '||' + _ds1['원본명'].astype(str)
_cu_gs_mask = _ds1['편의점명'].isin(['CU', 'GS25'])
_ds1.loc[_cu_gs_mask, 'IP_NM'] = _ds1.loc[_cu_gs_mask, '_key'].map(_cu_gs_map['IP_NM'])
_ds1.drop(columns=['_key'], inplace=True)

# 세븐일레븐
_seven_mask = _ds1['편의점명'] == '세븐일레븐'
_ds1.loc[_seven_mask, 'IP_NM'] = _ds1.loc[_seven_mask, '정규화명'].map(_seven_map)

# ── 저장 ────────────────────────────────────────────────────────────
_ds1.to_parquet(_OUT_DS1, index=False, engine='pyarrow')

_ds1_csv = _OUT_DS1.replace('.parquet', '.csv')
_ds1_export = _ds1.copy()
for _col in ['키워드', '키워드_정제']:
    if _col in _ds1_export.columns:
        _ds1_export[_col] = _ds1_export[_col].apply(
            lambda x: '|'.join(x) if isinstance(x, (list, np.ndarray)) and len(x) > 0 else ''
        )
_ds1_export.to_csv(_ds1_csv, index=False, encoding='utf-8-sig')
print(f'  CSV 저장: {_ds1_csv}')

print('[IP_NM 추가 완료]')
for _cvs in ['CU', 'GS25', '세븐일레븐']:
    _m = _ds1['편의점명'] == _cvs
    _filled = _ds1.loc[_m, 'IP_NM'].notna().sum()
    print(f'  {_cvs}: {_m.sum()}행 중 IP_NM 채워짐 {_filled}개 / NaN {_m.sum()-_filled}개')

del _cu_ip, _gs_ip, _cu_gs_map, _seven, _bridge, _seven_map


  CSV 저장: C:\Users\송정현\Documents\Projects\박재홍교수님세미나\Projects\20기\7eleven_npd_framework\data\processed\instagram_engagement_with_keywords.csv
[IP_NM 추가 완료]
  CU: 1459행 중 IP_NM 채워짐 555개 / NaN 904개
  GS25: 1489행 중 IP_NM 채워짐 269개 / NaN 1220개
  세븐일레븐: 1301행 중 IP_NM 채워짐 414개 / NaN 887개


## Phase 4E. Trend Engagement 데이터셋 빌드

**DS3** `trend_engagement_with_keywords.parquet`  
스키마: `트렌드(키워드) | 속성보유여부 | 속성(키워드) | 좋아요 수 | 언급일 | url | body`

**입력**
- `data/raw/knewnew_without_ad_with_keywords.csv`
- `data/raw/knewnew_1-01_04-22_with_keywords.csv`
- `data/processed/trend_keywords_processed.parquet`

**처리 흐름**
1. 두 CSV 병합 → 날짜·URL 기준 중복 제거 → `data/raw/knewnew_merged.csv`
2. `trend_keywords_processed.parquet`의 `트렌드_제품명` ↔ 포스트 `keywords` 매칭
3. 한 행 = (트렌드 키워드, 포스트) 쌍, `속성보유여부`는 해당 트렌드 키워드의 속성 존재 여부


In [31]:
# ── Phase 4E: Trend Engagement 데이터셋 빌드 ──────────────────────

_RAW_F1    = os.path.join(BASE_DIR, 'data', 'raw', 'knewnew_without_ad_with_keywords.csv')
_RAW_F2    = os.path.join(BASE_DIR, 'data', 'raw', 'knewnew_1-01_04-22_with_keywords.csv')
_TREND_PATH = TREND_PROC_PATH
_OUT_RAW   = os.path.join(BASE_DIR, 'data', 'raw', 'knewnew_merged.csv')
_OUT_DS3   = os.path.join(BASE_DIR, 'data', 'processed', 'trend_engagement_with_keywords.parquet')

_KEEP_COLS = ['date', 'body', 'likes', 'url', 'keywords']

def _load_knewnew(path):
    try:
        df = pd.read_csv(path, encoding='utf-8-sig')
    except UnicodeDecodeError:
        df = pd.read_csv(path, encoding='cp949')
    avail = [c for c in _KEEP_COLS if c in df.columns]
    df = df[avail].copy()
    df['date'] = pd.to_datetime(df['date'], errors='coerce').dt.strftime('%Y-%m-%d')
    return df

_df1 = _load_knewnew(_RAW_F1)
_df2 = _load_knewnew(_RAW_F2)
print(f'  파일1: {len(_df1)}행 ({_df1["date"].min()} ~ {_df1["date"].max()})')
print(f'  파일2: {len(_df2)}행 ({_df2["date"].min()} ~ {_df2["date"].max()})')

_merged = pd.concat([_df2, _df1], ignore_index=True)
_before = len(_merged)
_has_url  = _merged['url'].notna()
_merged = pd.concat([
    _merged[_has_url].drop_duplicates(subset='url', keep='first'),
    _merged[~_has_url].drop_duplicates(subset=['date', 'body'], keep='first'),
], ignore_index=True).sort_values('date').reset_index(drop=True)
print(f'  병합: {_before}행 → dedup 후: {len(_merged)}행')
_merged.to_csv(_OUT_RAW, index=False, encoding='utf-8-sig')
print(f'  원본 저장: {_OUT_RAW}')

_trend = pd.read_parquet(_TREND_PATH)
_trend_map = {
    row['트렌드_제품명']: row['확정키워드_정제']
    for _, row in _trend.iterrows()
}
_trend_names = set(_trend_map.keys())
print(f'  트렌드 키워드 수: {len(_trend_names)}개')

_rows = []
for _, _post in _merged.iterrows():
    _raw_kws = str(_post.get('keywords', '') or '')
    _post_kws = [k.strip() for k in _raw_kws.split(',') if k.strip()]
    for _kw in _post_kws:
        if _kw in _trend_names:
            _attr_kws = _trend_map[_kw]
            _rows.append({
                '트렌드(키워드)': _kw,
                '속성보유여부':   len(_attr_kws) > 0,
                '속성(키워드)':   _attr_kws,
                '좋아요 수':      _post.get('likes'),
                '언급일':         _post.get('date'),
                'url':            _post.get('url'),
                'body':           _post.get('body'),
            })

ds3 = pd.DataFrame(_rows)
_matched_posts = ds3['url'].nunique() if not ds3.empty else 0
print(f'  (트렌드키워드, 포스트) 쌍: {len(ds3):,}개 / 포스트 수: {_matched_posts:,}개')
if not ds3.empty:
    print(f'  트렌드 키워드 커버리지: {ds3["트렌드(키워드)"].nunique()}개 / {len(_trend_names)}개')

ds3.to_parquet(_OUT_DS3, index=False, engine='pyarrow')
print(f'  저장: {_OUT_DS3}')

_ds3_csv = _OUT_DS3.replace('.parquet', '.csv')
_ds3_export = ds3.copy()
if not _ds3_export.empty and '속성(키워드)' in _ds3_export.columns:
    _ds3_export['속성(키워드)'] = _ds3_export['속성(키워드)'].apply(lambda x: '|'.join(x) if isinstance(x, list) else '')
_ds3_export.to_csv(_ds3_csv, index=False, encoding='utf-8-sig')
print(f'  CSV 저장: {_ds3_csv}')

del _df1, _df2, _merged, _trend, _rows


  파일1: 1037행 (2025-04-21 ~ 2025-12-31)
  파일2: 471행 (2025-01-01 ~ 2025-04-21)
  병합: 1508행 → dedup 후: 1506행
  원본 저장: C:\Users\송정현\Documents\Projects\박재홍교수님세미나\Projects\20기\7eleven_npd_framework\data\raw\knewnew_merged.csv
  트렌드 키워드 수: 447개
  (트렌드키워드, 포스트) 쌍: 1,053개 / 포스트 수: 406개
  트렌드 키워드 커버리지: 430개 / 447개
  저장: C:\Users\송정현\Documents\Projects\박재홍교수님세미나\Projects\20기\7eleven_npd_framework\data\processed\trend_engagement_with_keywords.parquet
  CSV 저장: C:\Users\송정현\Documents\Projects\박재홍교수님세미나\Projects\20기\7eleven_npd_framework\data\processed\trend_engagement_with_keywords.csv


In [32]:
OUT_IP      = os.path.join(BASE_DIR, 'data', 'processed', 'ip_master_dataset.parquet')

# 트렌드 전처리 파일 로드 (Block 4C에서 저장)
df_trend_proc = pd.read_parquet(TREND_PROC_PATH)
print(f'트렌드 전처리 파일 로드: {df_trend_proc.shape}')

all_trend_attrs = set(
    kw
    for lst in df_trend_proc['확정키워드_정제']
    for kw in (lst.tolist() if isinstance(lst, np.ndarray) else lst if isinstance(lst, list) else [])
)
all_ip_names = set(
    ip
    for lst in df_trend_proc['ip_list']
    for ip in (lst.tolist() if isinstance(lst, np.ndarray) else lst if isinstance(lst, list) else [])
    if ip and ip != '없음'
)

print(f'  커뮤니티 수         : {df_trend_proc["군집ID"].nunique()}개')
print(f'  고유 트렌드 키워드  : {len(all_trend_attrs)}개')
print(f'  고유 IP/브랜드      : {len(all_ip_names)}개')


트렌드 전처리 파일 로드: (448, 6)
  커뮤니티 수         : 16개
  고유 트렌드 키워드  : 587개
  고유 IP/브랜드      : 202개


In [33]:
# ── Phase 5-3 준비 현황 ─────────────────────────────────────
print('=== Phase 5-3 준비: 트렌드 어휘 + IP/군집 구조 ===')
print(f'  df_trend_proc     : {len(df_trend_proc):,}행')
print(f'  고유 트렌드 키워드: {len(all_trend_attrs):,}개')
print(f'  고유 IP/브랜드    : {len(all_ip_names):,}개')


=== Phase 5-3 준비: 트렌드 어휘 + IP/군집 구조 ===
  df_trend_proc     : 448행
  고유 트렌드 키워드: 587개
  고유 IP/브랜드    : 202개


### Phase 5-3. IP Master 구축

`ip_name` 기준 GroupBy — 속성 없는 IP(빈 커뮤니티 소속)도 처리
- **스키마**: `ip_name`, `일반_속성`, `트렌드_속성`, `소속_커뮤니티`
- **출력**: `ip_master_dataset.parquet`

In [34]:
# ── Block 12: IP Master 구축 (키워드 raw 저장 → 분리는 01 Phase 5-2에서) ──
IP_KW_JSON = os.path.join(BASE_DIR, 'data', 'processed', 'IP_속성추출', 'IP_키워드사전.json')

if not os.path.exists(IP_KW_JSON):
    raise FileNotFoundError(f'IP_키워드사전.json 없음: {IP_KW_JSON}')

with open(IP_KW_JSON, 'r', encoding='utf-8') as f:
    ip_kw_list = json.load(f)
print(f'IP_키워드사전.json 로드: {len(ip_kw_list)}개 IP')

# 트렌드 데이터에서 IP별 소속 커뮤니티 매핑
_ip_comm_map = {}
for _, _row in df_trend_proc.iterrows():
    _ip_lst = _row['ip_list']
    _comm   = _row['군집명']
    _ip_lst = _ip_lst.tolist() if isinstance(_ip_lst, np.ndarray) else (_ip_lst or [])
    for _ip in _ip_lst:
        if _ip and _ip != '없음':
            _ip_comm_map.setdefault(_ip, set()).add(_comm)

# ip_name | 최종키워드 | 소속_커뮤니티 (트렌드/일반 분리는 01에서 수행)
_rows = []
for _entry in ip_kw_list:
    _ip     = _entry['ip_name']
    _ip_kws = sorted(_entry.get('최종키워드') or [])
    _comm   = sorted(_ip_comm_map.get(_ip, set()))
    _rows.append({'ip_name': _ip, '최종키워드': _ip_kws, '소속_커뮤니티': _comm})

ip_master = pd.DataFrame(_rows)

ip_master.to_parquet(OUT_IP, index=False, engine='pyarrow')
print(f'IP Master 저장: {OUT_IP}')

_ip_csv = OUT_IP.replace('.parquet', '.csv')
_ip_export = ip_master.copy()
_ip_export['최종키워드']   = _ip_export['최종키워드'].apply(lambda x: '|'.join(x) if isinstance(x, list) else '')
_ip_export['소속_커뮤니티'] = _ip_export['소속_커뮤니티'].apply(lambda x: '|'.join(x) if isinstance(x, list) else '')
_ip_export.to_csv(_ip_csv, index=False, encoding='utf-8-sig')
print(f'CSV 저장: {_ip_csv}')

print(f'shape: {ip_master.shape}')
print(f'키워드 있는 IP: {ip_master["최종키워드"].apply(len).gt(0).sum()}개')
ip_master.head(3)


IP_키워드사전.json 로드: 291개 IP
IP Master 저장: C:\Users\송정현\Documents\Projects\박재홍교수님세미나\Projects\20기\7eleven_npd_framework\data\processed\ip_master_dataset.parquet
CSV 저장: C:\Users\송정현\Documents\Projects\박재홍교수님세미나\Projects\20기\7eleven_npd_framework\data\processed\ip_master_dataset.csv
shape: (291, 3)
키워드 있는 IP: 287개


,ip_name,최종키워드,소속_커뮤니티
0,KBL,"[농구, 스포츠, 승부, 에너지, 열정, 짜릿함]",[]
1,최강록,"[깊음, 요리, 일식, 전문성, 진정성]",[편의점 시즌 콜라보]
2,추성훈,"[강함, 격투기, 문화, 이국적, 일본, 일상]",[이자카야·바 다이닝]


In [35]:
# ── Block 12 IP Master 현황 ─────────────────────────────────────
_kw_cnt = ip_master['최종키워드'].apply(lambda x: len(x) if isinstance(x, list) else 0)
print('=== Block 12 완료: ip_master_dataset.parquet ===')
print(f'  전체 IP 수         : {len(ip_master):,}개')
print(f'  키워드 있는 IP     : {(_kw_cnt > 0).sum():,}개')
print(f'  평균 키워드 수     : {_kw_cnt.mean():.1f}개/IP')
print('  ※ 트렌드/일반 분리는 01 Phase 5-2에서 수행')


=== Block 12 완료: ip_master_dataset.parquet ===
  전체 IP 수         : 291개
  키워드 있는 IP     : 287개
  평균 키워드 수     : 5.9개/IP
  ※ 트렌드/일반 분리는 01 Phase 5-2에서 수행


In [36]:
# ── Block 13: ip_keywords + trend_keywords 저장 ────────────────────────

# ── ip_keywords.parquet ─────────────────────────────────────────────────
IP_KW_JSON = os.path.join(BASE_DIR, 'data', 'processed', 'IP_속성추출', 'IP_키워드사전.json')
with open(IP_KW_JSON, 'r', encoding='utf-8') as f:
    _ip_kw_list = json.load(f)

df_ip_kw = pd.DataFrame([
    {'ip_name': e['ip_name'], '키워드': run_pipeline(e.get('최종키워드') or [])}
    for e in _ip_kw_list
])
IP_KW_PATH = os.path.join(BASE_DIR, 'data', 'processed', 'ip_keywords.parquet')
df_ip_kw.to_parquet(IP_KW_PATH, index=False)
print(f'ip_keywords.parquet: {len(df_ip_kw)}개 IP, 키워드 있는 IP: {(df_ip_kw["키워드"].apply(len) > 0).sum()}개')

# ── trend_keywords.parquet ───────────────────────────────────────────────
PRODUCT_ATTR_JSON = os.path.join(BASE_DIR, 'data', 'processed', 'IP_속성추출', '제품_속성.json')
with open(PRODUCT_ATTR_JSON, 'r', encoding='utf-8') as f:
    _prod_attrs = json.load(f)

_ATTR_COLS = ['flavor', 'texture', 'ingredients', 'tpo']
_prod_attr_map = {}
for _entry in _prod_attrs:
    _raw_kw = _entry.get('keyword', '').strip()
    if not _raw_kw:
        continue
    _norm_kws = run_pipeline([normalize_trend_name(_raw_kw)])
    _raw_attrs = []
    for _col in _ATTR_COLS:
        _val = _entry.get(_col, [])
        if isinstance(_val, list):
            _raw_attrs.extend(_val)
    if _entry.get('product_category'):
        _raw_attrs.append(_entry['product_category'])
    _norm_attrs = run_pipeline(_raw_attrs)
    for _nk in _norm_kws:
        _prod_attr_map.setdefault(_nk, set()).update(_norm_attrs)

_trend_kw_set_path = os.path.join(BASE_DIR, 'data', 'processed', 'trend_kw_set.json')
with open(_trend_kw_set_path, 'r', encoding='utf-8') as f:
    _trend_kw_set = set(json.load(f)['trend_kw_set'])

_trend_rows = []
for _kw in sorted(_trend_kw_set):
    _trend_rows.append({'트렌드_키워드': _kw, '추출_속성': sorted(_prod_attr_map.get(_kw, set()))})

df_trend_kw = pd.DataFrame(_trend_rows)
TREND_KW_PATH = os.path.join(BASE_DIR, 'data', 'processed', 'trend_keywords.parquet')
df_trend_kw.to_parquet(TREND_KW_PATH, index=False)
print(f'trend_keywords.parquet: {len(df_trend_kw)}개 트렌드 키워드')
print(f'  추출_속성 있음: {(df_trend_kw["추출_속성"].apply(len) > 0).sum()}개 / 빈 리스트: {(df_trend_kw["추출_속성"].apply(len) == 0).sum()}개')


ip_keywords.parquet: 291개 IP, 키워드 있는 IP: 287개
trend_keywords.parquet: 747개 트렌드 키워드
  추출_속성 있음: 450개 / 빈 리스트: 297개


---
### ip_keywords.parquet 통합 패치

Block 13 직후 실행. 키워드 제거·삭제·병합·수동 입력 등 모든 수동 편집을 일괄 적용.  
새 편집이 생기면 의 해당 섹션에만 추가.

In [37]:
# ── 통합 패치: ip_keywords.parquet 수동 편집 일괄 적용 ─────────────────────
import subprocess
_patch_path = os.path.join(BASE_DIR, "src", "data_builder", "patch_ip_keywords_all.py")
_result = subprocess.run(
    ["python", _patch_path],
    capture_output=True, text=True, encoding="utf-8"
)
print(_result.stdout)
if _result.returncode != 0:
    print(_result.stderr)


ip_keywords.parquet 통합 패치
로드: 291개 IP

[1] 키워드 제거
  라인프렌즈: ['힙함'] 제거
  이정후: ['화려함'] 제거
  마루짱: ['세련'] 제거
  NCTWISH: ['섬세'] 제거
  피스마이너스원: ['스트리머'] 제거
  꿈돌이: ['꿈', '돌'] 제거
  마츠시게유타카: ['이름', '존재'] 제거
  박종혁: ['배우', '정책'] 제거
  솔로지옥: ['미스터리'] 제거
  MLB: ['KBO'] 제거
  블루밍테일: ['KBO'] 제거

[2] IP 삭제
  무무도사 삭제
  리복방구 삭제
  최재승 삭제
  응답하라1985 삭제
  장채아 삭제

[3] 병합 / 이름 변경
  하츄핑 → 티니핑 병합 (추가: ['사랑', '하트', '프린세스'])
  캐치티니핑 → 티니핑 병합 (추가: ['티니핑', '공주', '반짝임'])
  온정돈까스 → 디진다 돈까스 병합 (추가: ['디진다', '바삭', '추억'])
  앙리마티스 → 앙티마티스 병합 (추가: ['야수파', '추상적', '순수함'])
  구마유시 → T1 병합 (추가: ['전문성', '집중력'])
  도란 → T1 병합 (추가: ['슈퍼', '스타', '경기력'])
  케리아 → T1 병합 (추가: ['LoL', '팀워크', 'T1'])
  어남선생 → 류수영 병합 (추가: ['배우', '요리', '따뜻', '유쾌'])
  엠즈 → 브롤스타즈 병합 (추가: ['향기', '스프레이', '스타', '압박', '슬로우', '좀비'])
  케데헌 → 케이팝데몬헌터스 병합 (추가: ['걸그룹', '애니메이션', '뮤지컬', '판타지', '한국', '문화', '팬덤'])
  카러플 → 카트라이더러쉬플러스 이름 변경
  투슬리스 → 드래곤길들이기 병합 (추가: [])

[4] 키워드 덮어쓰기
  다이노탱: ['귀여움', '캐릭터', '쿼카', '이미지', '콜라보', '마스코트']
  데르뜨: ['베이커리', '디저트', '크림', '달콤', '프리미엄', '감